# Name That LLM: TF-IDF + style features + fine-tuned DeBERTa + language-model features

Generated by `python -m src.build_notebook` from
https://github.com/Dana-Lev/name-that-llm. The cells below recreate the `src/`
package, then fit on `train.csv` and write `submission.parquet`.

In [ ]:
from pathlib import Path

for folder in ['src', 'src/features', 'src/models', 'src/segrun']:
    Path(folder).mkdir(parents=True, exist_ok=True)
for empty_file in ['src/__init__.py', 'src/features/__init__.py']:
    Path(empty_file).touch()

In [ ]:
%%writefile src/data.py
"""Loading the competition data and building the cross-validation folds."""

import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

# The 18 valid labels, grouped by lab. Order is fixed: column k of every
# probability matrix in this project refers to LABELS[k].
LABS = {
    "OpenAI": ["gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra"],
    "Google": [
        "gemini-3.1-pro-preview",
        "gemini-3.6-flash",
        "gemini-3.7-flash",
        "gemini-3.8-flash",
    ],
    "Anthropic": [
        "claude-fable-5",
        "claude-opus-4-6",
        "claude-opus-4-8",
        "claude-opus-5",
        "claude-sonnet-5",
    ],
    "Qwen": ["qwen3.8-max"],
    "ZAI": ["glm-5.3", "glm-5.3-flash"],
    "Moonshot AI": ["kimi-k3"],
}
LABELS = [model for models in LABS.values() for model in models]
LABEL_TO_ID = {label: i for i, label in enumerate(LABELS)}
LAB_OF = {model: lab for lab, models in LABS.items() for model in models}

N_SPLITS = 5
SEED = 42


def find_data_dir():
    """The folder with train.csv: $NTL_DATA_DIR, Kaggle's input folder, or ./data."""
    candidates = [
        os.environ.get("NTL_DATA_DIR"),
        "/kaggle/input/competitions/name-that-llm",
        "/kaggle/input/name-that-llm",
        "data",
    ]
    for candidate in candidates:
        if candidate and (Path(candidate) / "train.csv").exists():
            return Path(candidate)
    raise FileNotFoundError("train.csv not found; set NTL_DATA_DIR or put the data in ./data")


def load_train(data_dir="data"):
    """Read train.csv and add an integer `label` column."""
    df = pd.read_csv(Path(data_dir) / "train.csv")
    unknown = set(df["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"train.csv has labels outside the 18 valid ones: {sorted(unknown)}")
    df["label"] = df["model"].map(LABEL_TO_ID)
    return df


def load_test(data_dir="data"):
    return pd.read_csv(Path(data_dir) / "test.csv")


def make_folds(df, n_splits=N_SPLITS, seed=SEED):
    """Assign each row a fold id in [0, n_splits).

    All passages answering the same prompt land in the same fold, so every
    validation prompt is unseen during training, as in the hidden test set.
    Folds are also stratified by model to keep the class mix similar.
    """
    splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    folds = np.full(len(df), -1, dtype=int)
    for fold, (_, valid_idx) in enumerate(
        splitter.split(df["text"], df["label"], groups=df["prompt_id"])
    ):
        folds[valid_idx] = fold
    return folds


In [ ]:
%%writefile src/evaluate.py
"""Scoring: balanced accuracy (the competition metric) and confusion tables."""

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, confusion_matrix

from src.data import LAB_OF, LABELS


def balanced_accuracy(y_true, y_pred):
    """Mean per-class recall over the classes present in y_true."""
    return balanced_accuracy_score(y_true, y_pred)


def fold_scores(y_true, y_pred, folds):
    """Balanced accuracy of each validation fold."""
    return np.array(
        [balanced_accuracy(y_true[folds == f], y_pred[folds == f]) for f in np.unique(folds)]
    )


def model_confusion(y_true, y_pred):
    """18x18 table: rows are the true model, columns the predicted model."""
    ids = np.arange(len(LABELS))
    return pd.DataFrame(confusion_matrix(y_true, y_pred, labels=ids), index=LABELS, columns=LABELS)


def lab_confusion(y_true, y_pred):
    """The same table collapsed to the 6 labs."""
    table = model_confusion(y_true, y_pred)
    # An array, not a list: pandas reads a list of strings as column names.
    labs = np.array([LAB_OF[m] for m in LABELS])
    return table.groupby(labs, sort=False).sum().T.groupby(labs, sort=False).sum().T


def within_lab_error_share(y_true, y_pred):
    """Fraction of the mistakes where the predicted model is from the true lab."""
    table = lab_confusion(y_true, y_pred)
    errors = len(y_true) - int(np.sum(np.asarray(y_true) == np.asarray(y_pred)))
    if errors == 0:
        return 0.0
    model_table = model_confusion(y_true, y_pred).to_numpy()
    same_lab_total = int(np.trace(table.to_numpy()))
    same_lab_errors = same_lab_total - int(np.trace(model_table))
    return same_lab_errors / errors


In [ ]:
%%writefile src/features/tfidf.py
"""TF-IDF features over word and character n-grams."""

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion


def build_tfidf(word_ngrams=(1, 2), char_ngrams=(2, 5), min_df=2):
    """Word n-grams capture vocabulary; character n-grams capture punctuation
    and formatting habits. Case is kept because capitalisation is a style cue."""
    word = TfidfVectorizer(
        analyzer="word",
        ngram_range=word_ngrams,
        min_df=min_df,
        sublinear_tf=True,
        lowercase=False,
        token_pattern=r"(?u)\b\w+\b|[^\w\s]",
    )
    char = TfidfVectorizer(
        analyzer="char",
        ngram_range=char_ngrams,
        min_df=min_df,
        sublinear_tf=True,
        lowercase=False,
    )
    return FeatureUnion([("word", word), ("char", char)])


In [ ]:
%%writefile src/features/stylometry.py
"""Hand-made writing-style features: sentence shape, punctuation habits, formatting.

TF-IDF sees which words and character sequences appear. These features add what
it can't see: how long and how varied the sentences are, how the passage starts
and ends, and how often each punctuation mark is used per 100 characters.
"""

import re

import numpy as np

PUNCTUATION = {
    "comma": ",",
    "semicolon": ";",
    "colon": ":",
    "em_dash": "—",
    "en_dash": "–",
    "hyphen": "-",
    "open_paren": "(",
    "close_paren": ")",
    "straight_double_quote": '"',
    "straight_single_quote": "'",
    "curly_apostrophe": "’",
    "curly_open_quote": "“",
    "question": "?",
    "exclamation": "!",
    "ellipsis_char": "…",
    "three_dots": "...",
    "asterisk": "*",
    "backtick": "`",
    "hash": "#",
    "dollar": "$",
    "backslash": "\\",
    "slash": "/",
    "percent": "%",
    "equals": "=",
    "newline": "\n",
    "blank_line": "\n\n",
    "double_space": "  ",
}

SHAPE_NAMES = [
    "n_chars",
    "n_words",
    "n_sentences",
    "sentence_len_mean",
    "sentence_len_std",
    "sentence_len_max",
    "sentence_len_min",
    "word_len_mean",
    "word_len_std",
    "long_word_share",
    "distinct_word_share",
    "uppercase_share",
    "digit_share",
    "non_ascii_share",
    "starts_uppercase",
    "ends_with_sentence_mark",
    "ends_with_period",
    "pronoun_share",
    "n_list_items",
    "n_bold_spans",
    "n_latex_spans",
]
FEATURE_NAMES = SHAPE_NAMES + [f"{name}_per_100_chars" for name in PUNCTUATION]

_SENTENCE_END = re.compile(r"(?<=[.!?])\s+")
_PRONOUN = re.compile(r"\b(?:I|we|you)\b")
_LIST_ITEM = re.compile(r"^\s*(?:[-*•]|\d+[.)])\s", flags=re.M)
_BOLD = re.compile(r"\*\*[^*]+\*\*")
_LATEX = re.compile(r"\$[^$]+\$")


def style_vector(text):
    """The style features of one passage, in FEATURE_NAMES order."""
    words = text.split()
    sentences = [s for s in _SENTENCE_END.split(text) if s.strip()]
    sentence_lens = np.array([len(s.split()) for s in sentences] or [0])
    word_lens = np.array([len(w) for w in words] or [0])
    n_chars = max(len(text), 1)
    n_words = max(len(words), 1)
    last = text.rstrip()[-1:]

    shape = [
        len(text),
        len(words),
        len(sentences),
        sentence_lens.mean(),
        sentence_lens.std(),
        sentence_lens.max(),
        sentence_lens.min(),
        word_lens.mean(),
        word_lens.std(),
        (word_lens > 8).mean(),
        len({w.lower() for w in words}) / n_words,
        sum(c.isupper() for c in text) / n_chars,
        sum(c.isdigit() for c in text) / n_chars,
        sum(ord(c) > 127 for c in text) / n_chars,
        float(text[:1].isupper()),
        float(last in ".!?" and last != ""),
        float(last == "."),
        len(_PRONOUN.findall(text)) / n_words,
        len(_LIST_ITEM.findall(text)),
        len(_BOLD.findall(text)),
        len(_LATEX.findall(text)),
    ]
    punctuation = [text.count(mark) / n_chars * 100 for mark in PUNCTUATION.values()]
    return shape + punctuation


def style_features(texts):
    """Matrix of style features, one row per passage."""
    return np.array([style_vector(text) for text in texts], dtype=np.float64)


In [ ]:
%%writefile src/features/llm.py
"""Features read out of a frozen open language model (for example Qwen 3).

The model is not trained. Each passage is run through it once and we keep:

- "stats": 14 numbers on how predictable the passage is to the model
  (log-probability of each actual next token, entropy of the model's
  prediction, rank of the actual token among the model's guesses).
- "L<k>": the hidden vectors of layer k averaged over the passage, for four
  layers spread through the network.

This is heavy: run it on a GPU (Kaggle), not on a laptop.
"""

import json
import os
from pathlib import Path

import numpy as np

STAT_NAMES = [
    "logp_mean", "logp_std", "logp_q05", "logp_q25", "logp_q50", "logp_q75", "logp_q95",
    "entropy_mean", "entropy_std", "logp_plus_entropy_mean",
    "top1_share", "top5_share", "top20_share", "log_rank_mean",
]  # fmt: skip

_loaded = {}  # source -> (tokenizer, model, device), so the model loads once


def find_llm(root="/kaggle/input"):
    """Folder of the attached causal language model.

    $NTL_LLM_DIR wins. Otherwise the first folder under `root` whose
    config.json describes a ...ForCausalLM model and that has weights.
    """
    override = os.environ.get("NTL_LLM_DIR")
    if override:
        return override
    for config in sorted(Path(root).rglob("config.json")):
        folder = config.parent
        try:
            architectures = json.loads(config.read_text()).get("architectures") or []
        except (json.JSONDecodeError, OSError):
            continue
        has_weights = any(folder.glob("*.safetensors")) or (folder / "pytorch_model.bin").exists()
        if has_weights and any(a.endswith("ForCausalLM") for a in architectures):
            return str(folder)
    raise FileNotFoundError(f"no causal language model under {root}: attach one (e.g. Qwen 3) as an input")


def _load(source):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if source not in _loaded:
        device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(source)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"
        half = torch.float32 if device == "cpu" else torch.float16
        try:  # the argument was renamed from torch_dtype to dtype in newer transformers
            model = AutoModelForCausalLM.from_pretrained(source, dtype=half)
        except TypeError:
            model = AutoModelForCausalLM.from_pretrained(source, torch_dtype=half)
        model = model.to(device).eval()
        _loaded[source] = (tokenizer, model, device)
    return _loaded[source]


def unload():
    """Drop the loaded language models and free their GPU memory."""
    from src.models.transformer import free_gpu

    _loaded.clear()
    free_gpu()


def pick_layers(n_layers):
    return sorted({n_layers // 4, n_layers // 2, 3 * n_layers // 4, n_layers})


def extract(texts, source=None, batch_size=8, max_length=512, verbose=True, stats=True):
    """Return {"stats": (n, 14), "L<k>": (n, hidden_size), ...} for the passages.

    stats=False skips the predictability numbers, which are the slow part.
    """
    import torch

    texts = list(texts)
    source = find_llm() if source is None else source
    tokenizer, model, device = _load(source)
    layers = pick_layers(model.config.num_hidden_layers)
    features = {f"L{k}": np.zeros((len(texts), model.config.hidden_size), dtype=np.float32) for k in layers}
    features["stats"] = np.zeros((len(texts), len(STAT_NAMES)), dtype=np.float32)

    order = np.argsort([len(text) for text in texts], kind="stable")
    with torch.no_grad():
        for start in range(0, len(order), batch_size):
            rows = order[start : start + batch_size]
            # An empty passage would give an empty token row; one space keeps it valid.
            batch = [texts[i] if texts[i].strip() else " " for i in rows]
            encoded = tokenizer(
                batch, return_tensors="pt", padding=True, truncation=True, max_length=max_length
            ).to(device)
            out = model(**encoded, output_hidden_states=True)
            mask = encoded["attention_mask"]
            weights = mask.unsqueeze(-1).float()
            for k in layers:
                hidden = out.hidden_states[k].float()
                features[f"L{k}"][rows] = ((hidden * weights).sum(1) / weights.sum(1)).cpu().numpy()

            # Position t predicts token t+1. One passage at a time: the scores
            # over the whole vocabulary (150k words per token) for a full batch
            # don't fit in a 16 GB GPU.
            lengths = mask.sum(1).tolist()
            for j, i in enumerate(rows if stats else []):
                n = int(lengths[j])
                if n < 2:  # a one-token passage has nothing to predict
                    continue
                log_probs = torch.log_softmax(out.logits[j, : n - 1].float(), dim=-1)
                target = encoded["input_ids"][j, 1:n].unsqueeze(-1)
                actual = log_probs.gather(-1, target)
                lp = actual.squeeze(-1).cpu().numpy()
                en = (-(log_probs.exp() * log_probs).sum(-1)).cpu().numpy()
                rk = (log_probs > actual).sum(-1).cpu().numpy()
                del log_probs
                features["stats"][i] = [
                    lp.mean(), lp.std(), *np.quantile(lp, [0.05, 0.25, 0.5, 0.75, 0.95]),
                    en.mean(), en.std(), (lp + en).mean(),
                    (rk == 0).mean(), (rk < 5).mean(), (rk < 20).mean(), np.log1p(rk).mean(),
                ]  # fmt: skip
            if verbose and (start // batch_size) % 50 == 0:
                print(f"llm features: {start}/{len(texts)} passages", flush=True)
    return features


def early_layer_features(texts):
    """Averaged hidden vectors of the layer a quarter of the way up the model.

    In cross-validation this early layer named the model better than the
    middle or last layers (Qwen 3 4B: 0.379 against 0.336 and 0.316).
    """
    features = extract(texts, stats=False, verbose=False)
    # Freed after each use (fit, then predict) so another large model, such
    # as the LoRA-tuned Qwen, has the GPU to itself.
    unload()
    first_layer = min((name for name in features if name != "stats"), key=lambda name: int(name[1:]))
    return features[first_layer]


In [ ]:
%%writefile src/models/linear.py
"""Linear models: logistic regression on TF-IDF features and on style features."""

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from src.features.stylometry import style_features
from src.features.tfidf import build_tfidf


def build_tfidf_logreg(C=10.0, max_iter=3000, **tfidf_kwargs):
    """class_weight="balanced" matches the metric, which weights every model equally."""
    return Pipeline(
        [
            ("tfidf", build_tfidf(**tfidf_kwargs)),
            ("clf", LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced")),
        ]
    )


def build_style_logreg(C=1.0, max_iter=3000):
    """The style features are on very different scales, so standardise them first."""
    return Pipeline(
        [
            ("style", FunctionTransformer(style_features)),
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced")),
        ]
    )


def build_llm_logreg(C=0.01, max_iter=2000):
    """Logistic regression on features from a frozen language model (src/features/llm.py).

    Needs a GPU and the language model attached; see find_llm.
    """
    from src.features.llm import early_layer_features

    return Pipeline(
        [
            ("features", FunctionTransformer(early_layer_features)),
            ("scale", StandardScaler()),
            ("clf", LogisticRegression(C=C, max_iter=max_iter, class_weight="balanced")),
        ]
    )


In [ ]:
%%writefile src/models/blend.py
"""Combine several models by adding their log-probabilities."""

import time

import numpy as np

EPS = 1e-12


class LogProbBlend:
    """Weighted sum of the members' log-probabilities, renormalised.

    Adding log-probabilities multiplies probabilities, so a class scores high
    only if every member finds it plausible. A weight below 1 makes a member's
    opinion count for less.
    """

    def __init__(self, members):
        """members: list of (build_model, weight) pairs."""
        self.members = members

    def fit(self, texts, y):
        self.models_ = []
        for build, _ in self.members:
            started = time.time()
            print(f"blend: fitting {build.__name__} ...", flush=True)
            self.models_.append(build().fit(texts, y))
            print(f"blend: fitted {build.__name__} in {time.time() - started:.0f}s", flush=True)
        self.classes_ = self.models_[0].classes_
        for model, (build, _) in zip(self.models_, self.members):
            if not np.array_equal(model.classes_, self.classes_):
                raise ValueError(f"{build.__name__} knows classes {model.classes_}, not {self.classes_}")
        return self

    def predict_proba(self, texts):
        total = 0.0
        for model, (build, weight) in zip(self.models_, self.members):
            started = time.time()
            total = total + weight * np.log(model.predict_proba(texts) + EPS)
            print(f"blend: {build.__name__} predicted {len(texts)} passages in "
                  f"{time.time() - started:.0f}s", flush=True)  # fmt: skip
        total -= total.max(axis=1, keepdims=True)
        probs = np.exp(total)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, texts):
        return self.classes_[self.predict_proba(texts).argmax(axis=1)]


In [ ]:
%%writefile src/models/balance.py
"""Balance the predictions over the test passages, using no labels.

The blend picks some models far more often than others: on train.csv, which
has 150 passages of each model, it names gpt-5.6-luna 205 times and
claude-opus-5 only 103 times. If the test set is also about equal parts of
each model, one offset per model, added to the log-probabilities, can undo
that. `balance_offsets` finds the offsets that make the average predicted
probability of every model 1/18 over the passages being predicted. It uses
no labels, only the passages themselves, so it can run on the test set.

On train.csv, balancing each held-out fold on its own lifts the GLM blend
from 0.626 to 0.647 (better in 4 of 5 folds). It assumes balance: when the
models' counts in a fold were made to differ by up to 40%, the gain was
gone, and at 60% it hurt.
"""

import numpy as np

EPS = 1e-12


def balance_offsets(log_probs, iterations=200, step=0.5):
    """Offsets b such that softmax(log_probs + b) averages 1/k for each of the k classes."""
    k = log_probs.shape[1]
    offsets = np.zeros(k)
    for _ in range(iterations):
        logits = log_probs + offsets
        probs = np.exp(logits - logits.max(axis=1, keepdims=True))
        probs /= probs.sum(axis=1, keepdims=True)
        offsets -= step * np.log(probs.mean(axis=0) * k + EPS)
    return offsets


class Balanced:
    """A model whose test predictions are balanced with balance_offsets."""

    def __init__(self, build_base):
        self.build_base = build_base

    def fit(self, texts, y):
        self.base_ = self.build_base().fit(texts, y)
        self.classes_ = self.base_.classes_
        return self

    def predict_proba(self, texts):
        log_probs = np.log(self.base_.predict_proba(texts) + EPS)
        offsets = balance_offsets(log_probs)
        print("balance: offsets " + ", ".join(f"{c}:{b:+.2f}" for c, b in zip(self.classes_, offsets)),
              flush=True)  # fmt: skip
        logits = log_probs + offsets
        probs = np.exp(logits - logits.max(axis=1, keepdims=True))
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, texts):
        return self.classes_[self.predict_proba(texts).argmax(axis=1)]


In [ ]:
%%writefile src/models/transformer.py
"""Fine-tune a pretrained text encoder (DeBERTa-v3) to name the model.

The encoder reads the passage as tokens, we average its output vectors over
the passage and put one linear layer with 18 outputs on top. All weights are
trained, with a small learning rate for the pretrained part.

A large language model (Qwen 3) is fine-tuned the same way but with LoRA:
its own weights stay frozen in 16-bit precision, and each linear layer gets
a small trainable correction of low rank (see _lora_linear_class). Only those
corrections and the output layer are trained and saved.

torch and transformers are imported inside the class so the rest of the
project works without them.
"""

import json
import os
import time
from pathlib import Path

import numpy as np

DEFAULT_MODEL = "microsoft/deberta-v3-base"
MARKER_FILE = "ntl_encoder.json"  # written by FineTunedEncoder.save
HEAD_FILE = "ntl_head.pt"
LORA_FILE = "ntl_lora.pt"  # the LoRA weights of an encoder trained with lora_rank


def find_pretrained(name=DEFAULT_MODEL):
    """Where to load the pretrained weights from (`name` may also be a folder).

    Kaggle notebooks run without internet, so there the weights must be
    attached as an input; we look for a folder under /kaggle/input whose path
    contains the model's short name. Elsewhere the Hugging Face name is used
    (downloaded once, then cached). $NTL_PRETRAINED_DIR overrides both.
    """
    override = os.environ.get("NTL_PRETRAINED_DIR")
    if override:
        return override
    if Path(name).is_dir():
        return name
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        return find_weights_folder(kaggle_input, name)
    return name


def _squash(text):
    """Lower-case letters and digits only, so 'DeBERTa-v3-base' matches 'debertav3base'."""
    return "".join(ch for ch in text.lower() if ch.isalnum())


def find_weights_folder(root, name=DEFAULT_MODEL):
    """The folder under `root` that holds the pretrained model `name`.

    It must have a config.json next to a weights file, and its path must
    contain the model's short name (ignoring case and punctuation). An input
    that is merely called deberta-v3-base but has no weights doesn't count.
    """
    short = name.split("/")[-1]
    for config in sorted(Path(root).rglob("config.json")):
        folder = config.parent
        has_weights = any((folder / f).exists() for f in ("model.safetensors", "pytorch_model.bin"))
        if has_weights and _squash(short) in _squash(str(folder)):
            return str(folder)
    raise FileNotFoundError(
        f"no pretrained {short} weights under {root}: attach an input that contains "
        "config.json together with pytorch_model.bin or model.safetensors"
    )


_lora_class = None  # built on first use, so this module imports without torch


def _lora_linear_class():
    """A frozen nn.Linear plus a trainable low-rank correction.

    output = base(x) + (alpha / rank) * B(A(dropout(x))), with A of shape
    rank x in and B of shape out x rank. B starts at zero, so training starts
    from the pretrained model exactly. A and B are kept in 32-bit precision.
    """
    global _lora_class
    if _lora_class is not None:
        return _lora_class
    import math

    from torch import nn

    class _LoRALinear(nn.Module):
        def __init__(self, base, rank, alpha, dropout):
            super().__init__()
            self.base = base
            self.lora_a = nn.Linear(base.in_features, rank, bias=False)
            self.lora_b = nn.Linear(rank, base.out_features, bias=False)
            nn.init.kaiming_uniform_(self.lora_a.weight, a=math.sqrt(5))
            nn.init.zeros_(self.lora_b.weight)
            self.dropout = nn.Dropout(dropout)
            self.scale = alpha / rank

        def forward(self, x):
            correction = self.lora_b(self.lora_a(self.dropout(x.to(self.lora_a.weight.dtype))))
            return self.base(x) + (self.scale * correction).to(x.dtype)

    _lora_class = _LoRALinear
    return _lora_class


def add_lora(model, rank, alpha, dropout=0.05):
    """Freeze `model` and wrap every nn.Linear in it with LoRA.

    Returns the number of trainable weights added.
    """
    from torch import nn

    lora = _lora_linear_class()
    model.requires_grad_(False)
    targets = [
        (parent, name, child)
        for parent in model.modules()
        for name, child in parent.named_children()
        if isinstance(child, nn.Linear)
    ]
    for parent, name, child in targets:
        setattr(parent, name, lora(child, rank, alpha, dropout))
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def keep_first_layers(model, n):
    """Drop every transformer layer after the first n.

    For naming the model, early layers of a frozen Qwen 3 4B carried more
    signal than later ones (src/features/llm.py), and half the layers is half
    the compute.
    """
    model.layers = model.layers[:n]
    model.config.num_hidden_layers = n
    if getattr(model.config, "layer_types", None):
        model.config.layer_types = model.config.layer_types[:n]


def pick_device():
    import torch

    if torch.cuda.is_available():
        return "cuda"
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


class FineTunedEncoder:
    """Same interface as the scikit-learn models: fit, predict_proba, classes_."""

    def __init__(
        self,
        pretrained=DEFAULT_MODEL,
        epochs=6,
        lr=3e-5,
        head_lr_factor=20,
        batch_size=16,
        max_length=512,
        label_smoothing=0.05,
        seed=42,
        verbose=False,
        fast=False,
        progress_every=0,
        time_limit_hours=None,
        balanced=False,
        lora_rank=None,
        lora_alpha=32,
        keep_layers=None,
        gradient_checkpointing=False,
    ):
        """fast=True is for large training sets on a GPU: 16-bit arithmetic
        and batches of passages of similar length. progress_every=N prints
        the loss and the time left every N steps. time_limit_hours stops
        training early, so a Kaggle run (killed at 12 hours) still saves.
        balanced=True weights the loss by 1 / (passages of that model), so
        models with few training passages count as much as the others, as
        they do in balanced accuracy.
        lora_rank=r trains a LoRA correction of rank r on a frozen 16-bit
        model (for a large language model) instead of all weights;
        keep_layers=n keeps only its first n layers. gradient_checkpointing
        recomputes activations in the backward pass to save GPU memory."""
        self.pretrained = pretrained
        self.epochs = epochs
        self.lr = lr
        self.head_lr_factor = head_lr_factor
        self.batch_size = batch_size
        self.max_length = max_length
        self.label_smoothing = label_smoothing
        self.seed = seed
        self.verbose = verbose
        self.fast = fast
        self.progress_every = progress_every
        self.time_limit_hours = time_limit_hours
        self.balanced = balanced
        self.lora_rank = lora_rank
        self.lora_alpha = lora_alpha
        self.keep_layers = keep_layers
        self.gradient_checkpointing = gradient_checkpointing

    def _load_network(self, source):
        """Tokenizer and encoder from `source`, on self.device_."""
        import torch
        from transformers import AutoModel, AutoTokenizer

        self.tokenizer_ = AutoTokenizer.from_pretrained(source)
        if self.tokenizer_.pad_token is None:
            self.tokenizer_.pad_token = self.tokenizer_.eos_token
        self.tokenizer_.padding_side = "right"  # mean pooling uses the mask, so either side works
        if not self.lora_rank:
            # .float(): the published DeBERTa-v3 weights are stored in 16-bit
            # precision, and training all of them in 16 bits turns the loss into NaN.
            self.encoder_ = AutoModel.from_pretrained(source).float().to(self.device_)
            return
        dtype = torch.float16 if self.device_ == "cuda" else torch.float32
        try:  # the argument was renamed from torch_dtype to dtype in newer transformers
            encoder = AutoModel.from_pretrained(source, dtype=dtype)
        except TypeError:
            encoder = AutoModel.from_pretrained(source, torch_dtype=dtype)
        if self.keep_layers:
            keep_first_layers(encoder, self.keep_layers)
        trainable = add_lora(encoder, self.lora_rank, self.lora_alpha)
        if self.gradient_checkpointing:
            encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        if self.verbose:
            total = sum(p.numel() for p in encoder.parameters())
            print(f"LoRA: training {trainable:,} of {total:,} weights", flush=True)
        self.encoder_ = encoder.to(self.device_)

    def _autocast(self):
        import contextlib

        import torch

        if self.fast and self.device_ == "cuda":
            return torch.autocast("cuda", dtype=torch.float16)
        return contextlib.nullcontext()

    def _training_order(self, texts, rng):
        """Shuffled row order for one epoch.

        With fast=True, rows are shuffled, cut into chunks of 64 batches,
        sorted by length inside each chunk, and the batches shuffled again:
        still random, but with little padding.
        """
        order = rng.permutation(len(texts))
        if not self.fast:
            return order
        chunk = self.batch_size * 64
        batches = []
        for start in range(0, len(order), chunk):
            part = order[start : start + chunk]
            part = part[np.argsort([len(texts[i]) for i in part], kind="stable")]
            batches += [part[i : i + self.batch_size] for i in range(0, len(part), self.batch_size)]
        return np.concatenate([batches[i] for i in rng.permutation(len(batches))])

    def _batches(self, texts, order):
        import torch

        for start in range(0, len(order), self.batch_size):
            rows = order[start : start + self.batch_size]
            encoded = self.tokenizer_(
                [texts[i] for i in rows],
                truncation=True,
                max_length=self.max_length,
                padding=True,
                return_tensors="pt",
            )
            yield (
                rows,
                encoded["input_ids"].to(self.device_),
                encoded["attention_mask"].to(self.device_),
            )

    def _logits(self, ids, mask):
        hidden = self.encoder_(input_ids=ids, attention_mask=mask).last_hidden_state
        weights = mask.unsqueeze(-1).float()
        # Mean over real tokens only; padding has mask 0.
        pooled = (hidden * weights).sum(1) / weights.sum(1)
        return self.head_(self.dropout_(pooled))

    def fit(self, texts, y):
        import torch
        from torch import nn
        from transformers import get_cosine_schedule_with_warmup

        texts = list(texts)
        y = np.asarray(y)
        self.classes_ = np.unique(y)
        targets = np.searchsorted(self.classes_, y)

        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        self.device_ = pick_device()
        self._load_network(find_pretrained(self.pretrained))
        self.dropout_ = nn.Dropout(0.1)
        self.head_ = nn.Linear(self.encoder_.config.hidden_size, len(self.classes_)).to(self.device_)

        # The new layer starts from random weights, so it gets a larger step
        # than the pretrained encoder.
        # With LoRA only the corrections are trainable; the frozen weights are left out.
        trainable = [p for p in self.encoder_.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(
            [
                {"params": trainable, "lr": self.lr},
                {"params": self.head_.parameters(), "lr": self.lr * self.head_lr_factor},
            ],
            weight_decay=0.01,
        )
        steps = self.epochs * int(np.ceil(len(texts) / self.batch_size))
        schedule = get_cosine_schedule_with_warmup(optimizer, int(0.1 * steps), steps)
        parameters = trainable + list(self.head_.parameters())
        # 16-bit arithmetic needs the loss scaled up so small gradients survive.
        scaler = torch.amp.GradScaler("cuda", enabled=self.fast and self.device_ == "cuda")
        class_weights = None
        if self.balanced:
            counts = np.bincount(targets, minlength=len(self.classes_))
            # Mean weight 1 over the passages, so the learning rate means the same.
            class_weights = torch.tensor(len(targets) / (len(counts) * counts), dtype=torch.float32)
            class_weights = class_weights.to(self.device_)

        started = time.time()
        step = 0
        for epoch in range(self.epochs):
            self.encoder_.train()
            self.dropout_.train()
            total, seen, recent = 0.0, 0, []
            for rows, ids, mask in self._batches(texts, self._training_order(texts, rng)):
                target = torch.tensor(targets[rows]).to(self.device_)
                with self._autocast():
                    logits = self._logits(ids, mask)
                loss = nn.functional.cross_entropy(
                    logits.float(), target, weight=class_weights, label_smoothing=self.label_smoothing
                )
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(parameters, 1.0)
                scaler.step(optimizer)
                scaler.update()
                schedule.step()
                optimizer.zero_grad()
                total += loss.item() * len(rows)
                seen += len(rows)
                recent.append(loss.item())
                step += 1

                elapsed = time.time() - started
                if self.progress_every and step % self.progress_every == 0:
                    left = elapsed / step * (steps - step)
                    print(f"step {step}/{steps} (epoch {epoch + 1}): loss {np.mean(recent):.3f}, "
                          f"{elapsed / 60:.0f} min so far, about {left / 60:.0f} min left", flush=True)  # fmt: skip
                    recent = []
                if self.time_limit_hours and elapsed > self.time_limit_hours * 3600:
                    print(f"time limit of {self.time_limit_hours} h reached at step {step}/{steps}: "
                          "stopping early", flush=True)  # fmt: skip
                    return self
            if self.verbose:
                print(f"epoch {epoch + 1}/{self.epochs}: training loss {total / seen:.3f}", flush=True)
        return self

    def save(self, folder):
        """Write everything predict_proba needs; load it back with load_encoder."""
        import torch

        folder = Path(folder)
        folder.mkdir(parents=True, exist_ok=True)
        if self.lora_rank:
            # Only the LoRA weights: the frozen model is loaded again from its own input.
            lora = {k: v for k, v in self.encoder_.state_dict().items() if ".lora_" in k}
            torch.save(lora, folder / LORA_FILE)
        else:
            self.encoder_.save_pretrained(folder)
            self.tokenizer_.save_pretrained(folder)
        torch.save(self.head_.state_dict(), folder / HEAD_FILE)
        settings = {"classes": self.classes_.tolist(), "batch_size": self.batch_size,
                    "max_length": self.max_length, "fast": self.fast, "lora_rank": self.lora_rank,
                    "lora_alpha": self.lora_alpha, "keep_layers": self.keep_layers}  # fmt: skip
        (folder / MARKER_FILE).write_text(json.dumps(settings))
        return folder

    def predict_proba(self, texts):
        import torch

        texts = list(texts)
        self.encoder_.eval()
        self.dropout_.eval()
        probs = np.zeros((len(texts), len(self.classes_)))
        # Similar lengths in one batch means less padding, so faster.
        order = np.argsort([len(text) for text in texts], kind="stable")
        with torch.no_grad():
            for rows, ids, mask in self._batches(texts, order):
                with self._autocast():
                    logits = self._logits(ids, mask)
                probs[rows] = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        return probs

    def predict(self, texts):
        return self.classes_[self.predict_proba(texts).argmax(axis=1)]


def free_gpu():
    """Return the memory of models no longer referenced to the GPU."""
    import gc

    import torch

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_encoder(folder, base=None):
    """A FineTunedEncoder saved with .save(), ready to predict.

    A LoRA model also needs its frozen base model: `base`, or else the
    language model attached as an input (src.features.llm.find_llm).
    """
    import torch
    from torch import nn

    folder = Path(folder)
    settings = json.loads((folder / MARKER_FILE).read_text())
    lora_rank = settings.get("lora_rank")
    model = FineTunedEncoder(
        str(folder), batch_size=settings["batch_size"], max_length=settings["max_length"],
        fast=settings["fast"], lora_rank=lora_rank, lora_alpha=settings.get("lora_alpha", 32),
        keep_layers=settings.get("keep_layers"),
    )  # fmt: skip
    model.classes_ = np.array(settings["classes"])
    model.device_ = pick_device()
    if lora_rank:
        from src.features.llm import find_llm

        model._load_network(base or find_llm())
        lora = torch.load(folder / LORA_FILE, map_location="cpu")
        missing = set(lora) - set(model.encoder_.state_dict())
        assert not missing, f"LoRA weights with no place in the base model: {sorted(missing)[:3]}"
        model.encoder_.load_state_dict(lora, strict=False)
    else:
        model._load_network(folder)
    model.dropout_ = nn.Dropout(0.1)
    model.head_ = nn.Linear(model.encoder_.config.hidden_size, len(model.classes_))
    model.head_.load_state_dict(torch.load(folder / HEAD_FILE, map_location="cpu"))
    model.head_.to(model.device_)
    return model


def find_saved_encoder(root="/kaggle/input", name=None):
    """Folder of an encoder saved by .save(): $NTL_ENCODER_DIR, or the only one under `root`.

    `name` is the folder the training run saved into (deberta_pool_long,
    deberta_pool_big, ...); with it, only that encoder counts, so several
    can be attached at once. Without it, two saved encoders under `root` is
    an error rather than a guess, so a submission never silently loads the
    wrong one.
    """
    override = os.environ.get("NTL_ENCODER_DIR")
    if override and name is None:
        if not (Path(override) / MARKER_FILE).exists():
            raise FileNotFoundError(f"$NTL_ENCODER_DIR={override} has no {MARKER_FILE}")
        return Path(override)
    hits = sorted(Path(root).rglob(MARKER_FILE))
    if name is not None:
        hits = [hit for hit in hits if hit.parent.name == name]
    if not hits:
        what = f"saved encoder {name}" if name else f"saved encoder ({MARKER_FILE})"
        raise FileNotFoundError(
            f"no {what} under {root}: attach the output of the notebook that trained it "
            "(train_deberta_pool*.ipynb) as an input"
        )
    if len(hits) > 1:
        found = ", ".join(str(hit.parent) for hit in hits)
        raise ValueError(f"more than one saved encoder under {root} ({found}): attach only one")
    return hits[0].parent


class SavedEncoder:
    """An encoder trained in another notebook. fit() only loads it.

    The training passages given to fit are ignored: the model already learned
    from the open pool. This lets a submission use a model that takes hours
    to train. `name` picks one encoder when several are attached (see
    find_saved_encoder).
    """

    def __init__(self, name=None):
        self.name = name

    def fit(self, texts, y):
        self.folder_ = find_saved_encoder(name=self.name)
        settings = json.loads((self.folder_ / MARKER_FILE).read_text())
        self.classes_ = np.array(settings["classes"])
        return self

    def predict_proba(self, texts):
        # Loaded only while it predicts, then freed: a submission can use
        # several large models (two Qwen 3 4B among them), and a T4 has 15 GB.
        model = load_encoder(self.folder_)
        try:
            return model.predict_proba(texts)
        finally:
            del model
            free_gpu()

    def predict(self, texts):
        return self.classes_[self.predict_proba(texts).argmax(axis=1)]


def build_deberta_base(**kwargs):
    return FineTunedEncoder("microsoft/deberta-v3-base", lr=3e-5, **kwargs)


def build_deberta_small(**kwargs):
    return FineTunedEncoder("microsoft/deberta-v3-small", lr=5e-5, **kwargs)


In [ ]:
%%writefile src/pool.py
"""The public "open pool": extra labelled passages generated by other participants.

Dataset: https://www.kaggle.com/datasets/crtproduction/name-that-llm-open-pool
(CC BY-SA 4.0, about 226,000 passages from the 18 models, gathered from 65
public datasets). The competition rules (section 6) allow public external
data. The pool's prompts are not the competition's: it has no answers to the
prompts in train.csv, so train.csv can serve as a clean test set for a model
trained on the pool alone.

Labels are what each source claims; nobody re-verified which model wrote a
passage.
"""

import os
from pathlib import Path

import pandas as pd

from src.data import LABEL_TO_ID

POOL_FILE = "ntl_open_pool.parquet"
COLUMNS = ["text", "model", "match_weight"]


def find_pool(root="/kaggle/input"):
    """Path of the pool file: $NTL_POOL_FILE, or the first match under `root`."""
    override = os.environ.get("NTL_POOL_FILE")
    if override:
        return Path(override)
    hits = sorted(Path(root).rglob(POOL_FILE))
    if not hits:
        raise FileNotFoundError(f"{POOL_FILE} not found under {root}: attach the open pool dataset")
    return hits[0]


def load_pool(path=None):
    """Read the pool with an integer `label` column, like load_train."""
    pool = pd.read_parquet(find_pool() if path is None else path, columns=COLUMNS)
    pool = pool[pool["model"].isin(LABEL_TO_ID)].reset_index(drop=True)
    pool["label"] = pool["model"].map(LABEL_TO_ID)
    return pool


def sample_pool(pool, per_label, matched=False, seed=0):
    """At most `per_label` passages of each model.

    matched=True draws with the pool's `match_weight`, which favours passages
    whose punctuation and length look like the real ones of that model.
    """
    parts = []
    for _, group in pool.groupby("label"):
        n = min(per_label, len(group))
        weights = group["match_weight"].fillna(0) if matched else None
        if matched and (weights > 0).sum() < n:
            weights = None  # not enough weighted rows to draw from
        parts.append(group.sample(n, weights=weights, random_state=seed))
    return pd.concat(parts).reset_index(drop=True)


In [ ]:
%%writefile src/models/pool_model.py
"""Models trained on train.csv plus the public open pool (src/pool.py)."""

import numpy as np
import pandas as pd

from src.models.linear import build_tfidf_logreg
from src.pool import load_pool, sample_pool


def build_big_tfidf_logreg():
    """TF-IDF + logistic regression sized for 100,000+ passages.

    The vocabulary and the number of optimiser steps are capped so the fit
    takes minutes, not hours.
    """
    model = build_tfidf_logreg(max_iter=300, min_df=3)
    for _, vectorizer in model.named_steps["tfidf"].transformer_list:
        vectorizer.set_params(max_features=300_000, dtype=np.float32)
    return model


class PoolAugmented:
    """Fit a model on the given passages plus a sample of the open pool.

    Same interface as the other models (fit, predict_proba, classes_). The
    pool sample is at most `per_label` passages of each model.
    """

    def __init__(self, build_model=build_big_tfidf_logreg, per_label=8000, matched=False):
        self.build_model = build_model
        self.per_label = per_label
        self.matched = matched

    def fit(self, texts, y):
        extra = sample_pool(load_pool(), self.per_label, self.matched)
        all_texts = pd.concat([pd.Series(list(texts)), extra["text"]], ignore_index=True)
        all_labels = np.concatenate([np.asarray(y), extra["label"].to_numpy()])
        self.model_ = self.build_model().fit(all_texts, all_labels)
        self.classes_ = self.model_.classes_
        return self

    def predict_proba(self, texts):
        return self.model_.predict_proba(texts)

    def predict(self, texts):
        return self.model_.predict(texts)


def build_pool_tfidf_logreg():
    return PoolAugmented()


In [ ]:
%%writefile src/models/glm_model.py
"""A blend, then the likelihood under the GLM models for the passages it thinks may be GLM's.

Scoring a passage under both GLM models costs about 0.03 s per token on
Kaggle's T4 x2, plus about 53 minutes of reading the weights per run, so all
~4,000 test passages would take about 6.5 hours. Instead the blend runs
first, and only the passages to which it gives the most glm-5.3 plus
glm-5.3-flash probability are scored: at most `fraction` of them, fewer if
the time left would not cover them. On train.csv the top 30% hold 96% of the
real GLM passages, and adding the GLM classifier's log-probabilities to those
at weight 1.5 lifts the blend from 0.592 to 0.626 (glm-5.3 recall 0.20 to
0.57, glm-5.3-flash 0.37 to 0.56).

The GLM classifier is a logistic regression on the likelihood features. It is
fitted on the train.csv features that train_glm_likelihood.ipynb saved
(glm_<model>_train.csv, attach that notebook's output), so train.csv is not
scored again. If the GLM run fails or runs out of time, the blend's own
predictions stand.
"""

import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.data import LABELS, find_data_dir, load_train

STARTED = time.time()  # about when the notebook started: src.models is imported first thing
EPS = 1e-12
GLM_LABELS = [LABELS.index("glm-5.3"), LABELS.index("glm-5.3-flash")]
# seconds per run (reading the weights) and per token row, measured on train.csv
COST = {"glm53_flash": (1350.0, 0.0113), "glm53": (1820.0, 0.0199)}


def find_glm_train(model, root="/kaggle/input"):
    """The train.csv features of `model` saved by train_glm_likelihood.ipynb."""
    hits = sorted(Path(root).rglob(f"glm_{model}_train.csv"))
    if not hits:
        raise FileNotFoundError(
            f"no glm_{model}_train.csv under {root}: attach the output of ntl-train-glm-likelihood"
        )
    return hits[0]


def token_rows(texts):
    """Rows ntl-segrun will process per passage: the prompt prefix plus about 1.3 tokens per word."""
    words = np.array([len(str(text).split()) for text in texts])
    return 13 + np.minimum(320, np.ceil(1.35 * words))


def affordable(rows, models, seconds):
    """How many of the passages (in the given order) all `models` can score in `seconds`."""
    fixed = sum(COST[m][0] for m in models)
    per_row = sum(COST[m][1] for m in models)
    if seconds <= fixed:
        return 0
    return int(np.searchsorted(np.cumsum(rows) * per_row, seconds - fixed, side="right"))


class GatedGLMBlend:
    def __init__(self, build_base, weight=1.5, fraction=0.3, C=0.1, hours=8.0, margin_s=1800):
        self.build_base = build_base
        self.weight = weight
        self.fraction = fraction
        self.C = C
        self.hours = hours  # Kaggle stops a submission at 9 hours
        self.margin_s = margin_s

    def fit(self, texts, y):
        from src.features.glm import SEGMENTS
        from src.glm_experiment import feature_matrix

        self.base_ = self.build_base().fit(texts, y)
        self.classes_ = self.base_.classes_
        train = load_train(find_data_dir())
        assert list(train["text"]) == list(texts), "the GLM features are those of train.csv, in its order"
        ids = train["id"].astype(str)
        tables = {m: pd.read_csv(find_glm_train(m), dtype={"id": str}).set_index("id").reindex(ids)
                  for m in SEGMENTS}  # fmt: skip
        # one classifier on both models, and one on Flash alone in case GLM-5.3 does not finish
        self.classifiers_ = {}
        for models in (tuple(SEGMENTS), ("glm53_flash",)):
            X = feature_matrix({m: tables[m] for m in models})
            classifier = make_pipeline(
                StandardScaler(), LogisticRegression(C=self.C, max_iter=3000, class_weight="balanced")
            )
            classifier.fit(X.to_numpy(), np.asarray(y))
            assert np.array_equal(classifier.classes_, self.classes_)
            self.classifiers_[models] = (classifier, list(X.columns))
        return self

    def predict_proba(self, texts):
        from src.features.glm import SEGMENTS, score
        from src.glm_experiment import feature_matrix

        texts = list(texts)
        probs = self.base_.predict_proba(texts)
        log_probs = np.log(probs + EPS)

        mass = probs[:, GLM_LABELS].sum(axis=1)
        order = np.argsort(-mass, kind="stable")
        wanted = max(1, int(round(self.fraction * len(texts))))
        rows = token_rows([texts[i] for i in order])
        seconds = self.hours * 3600 - (time.time() - STARTED) - self.margin_s
        count = min(wanted, affordable(rows, tuple(SEGMENTS), seconds))
        models = tuple(SEGMENTS)
        if count == 0:
            count, models = min(wanted, affordable(rows, ("glm53_flash",), seconds)), ("glm53_flash",)
        print(f"GLM: scoring {count} of {len(texts)} passages under {models} "
              f"({seconds / 3600:.1f} h left)", flush=True)  # fmt: skip
        if count == 0:
            return probs
        chosen = order[:count]
        ids = [str(i) for i in chosen]

        tables = {}
        for model in models:
            budget = self.hours * 3600 - (time.time() - STARTED) - self.margin_s / 2
            try:
                tables[model] = score([texts[i] for i in chosen], model, "glm_test", ids=ids,
                                      budget_s=budget).reindex(ids)  # fmt: skip
            except Exception as error:
                print(f"GLM: {model} failed ({error!r}); using what finished", flush=True)
                break
        key = tuple(m for m in models if m in tables)
        if key not in self.classifiers_:
            print("GLM: no usable run; the blend's predictions stand", flush=True)
            return probs
        classifier, columns = self.classifiers_[key]
        X = feature_matrix({m: tables[m] for m in key})[columns]
        complete = X.notna().all(axis=1).to_numpy()
        rows_done = chosen[complete]
        glm = classifier.predict_proba(X.to_numpy()[complete])
        log_probs[rows_done] += self.weight * np.log(glm + EPS)
        print(f"GLM: added to {len(rows_done)} passages", flush=True)
        log_probs -= log_probs.max(axis=1, keepdims=True)
        out = np.exp(log_probs)
        return out / out.sum(axis=1, keepdims=True)

    def predict(self, texts):
        return self.classes_[self.predict_proba(texts).argmax(axis=1)]


In [ ]:
%%writefile src/models/__init__.py
"""Every model that can be cross-validated or submitted, by name."""

from src.models.balance import Balanced
from src.models.blend import LogProbBlend
from src.models.glm_model import GatedGLMBlend
from src.models.linear import build_llm_logreg, build_style_logreg, build_tfidf_logreg
from src.models.pool_model import build_pool_tfidf_logreg
from src.models.transformer import SavedEncoder, build_deberta_base, build_deberta_small


def build_tfidf_style_blend(style_weight=0.5):
    return LogProbBlend([(build_tfidf_logreg, 1.0), (build_style_logreg, style_weight)])


def build_tfidf_style_deberta_blend(style_weight=0.5, deberta_weight=0.5):
    return LogProbBlend(
        [
            (build_tfidf_logreg, 1.0),
            (build_style_logreg, style_weight),
            (build_deberta_base, deberta_weight),
        ]
    )


def build_full_blend(style_weight=0.5, deberta_weight=0.5, llm_weight=0.5):
    return LogProbBlend(
        [
            (build_tfidf_logreg, 1.0),
            (build_style_logreg, style_weight),
            (build_deberta_base, deberta_weight),
            (build_llm_logreg, llm_weight),
        ]
    )


def build_pool_blend(style_weight=0.5, deberta_weight=0.5, llm_weight=0.5):
    """The full blend with TF-IDF trained on train.csv plus the open pool."""
    return LogProbBlend(
        [
            (build_pool_tfidf_logreg, 1.0),
            (build_style_logreg, style_weight),
            (build_deberta_base, deberta_weight),
            (build_llm_logreg, llm_weight),
        ]
    )


def build_pool_deberta_blend(style_weight=0.5, deberta_weight=1.0, llm_weight=0.5):
    """TF-IDF and DeBERTa both trained on the open pool.

    The DeBERTa was trained in train_deberta_pool.ipynb and is only loaded
    here, so that notebook's output must be attached. Weights from
    src.blend_oof: the best ones lie on a flat plateau (0.573-0.576), and the
    DeBERTa trained on train.csv adds nothing once this one is in.
    """
    return LogProbBlend(
        [
            (build_pool_tfidf_logreg, 1.0),
            (build_style_logreg, style_weight),
            (SavedEncoder, deberta_weight),
            (build_llm_logreg, llm_weight),
        ]
    )


def load_deberta_pool_long():
    """DeBERTa trained on 136,000 pool passages (train_deberta_pool_long.ipynb)."""
    return SavedEncoder("deberta_pool_long")


def load_deberta_pool_big():
    """DeBERTa trained on 194,000 pool passages with a per-model loss weight
    (train_deberta_pool_big.ipynb)."""
    return SavedEncoder("deberta_pool_big")


def build_pool_two_deberta_blend(style_weight=0.5, long_weight=1.0, big_weight=1.0, llm_weight=0.5):
    """Both pool DeBERTas, which make different mistakes: the big one,
    trained with a per-model loss weight, finds the rare gemini-3.7-flash far
    more often (recall 0.44 against 0.26) and gemini-3.8-flash less often.
    Each alone in the blend scores 0.578 and 0.586, the two together 0.592
    (src.blend_oof). Needs the outputs of both training notebooks attached.
    """
    return LogProbBlend(
        [
            (build_pool_tfidf_logreg, 1.0),
            (build_style_logreg, style_weight),
            (load_deberta_pool_long, long_weight),
            (load_deberta_pool_big, big_weight),
            (build_llm_logreg, llm_weight),
        ]
    )


def load_qwen_lora_pool():
    """Qwen 3 4B, first 18 layers, LoRA-tuned on 71,500 pool passages
    (train_qwen_lora_pool.ipynb). Loads on top of the attached Qwen 3 4B."""
    return SavedEncoder("qwen_lora_pool")


def build_pool_qwen_lora_blend(
    style_weight=0.5, long_weight=0.5, big_weight=1.0, llm_weight=0.5, lora_weight=1.5
):
    """The two-DeBERTa blend plus the LoRA-tuned Qwen, our best single model
    (0.547 on train.csv). Adding it at weight 1.5 lifts the blend from 0.592
    to 0.604; halving the longer DeBERTa, which errs much like the big one,
    gives 0.609 (src.blend_oof). Needs the outputs of three training
    notebooks attached.
    """
    return LogProbBlend(
        [
            (build_pool_tfidf_logreg, 1.0),
            (build_style_logreg, style_weight),
            (load_deberta_pool_long, long_weight),
            (load_deberta_pool_big, big_weight),
            (load_qwen_lora_pool, lora_weight),
            (build_llm_logreg, llm_weight),
        ]
    )


def build_pool_qwen_lora_light_blend():
    """The 0.581 two-DeBERTa blend unchanged, plus the LoRA Qwen at weight 0.5
    (0.599 on train.csv). At weight 1.5, with the longer DeBERTa halved, the
    blend scored 0.609 on train.csv but 0.570 on the board, while the LoRA Qwen
    alone held up (0.547 and 0.537). This checks whether a light touch helps.
    """
    return build_pool_qwen_lora_blend(long_weight=1.0, lora_weight=0.5)


def build_pool_glm_blend():
    """The 0.581 two-DeBERTa blend, then the GLM likelihood on the 30% of test
    passages it thinks most likely GLM's (src/models/glm_model.py): 0.626 on
    train.csv against 0.592. Needs, besides that blend's inputs, the output of
    ntl-train-glm-likelihood, the output of estopadilla/ntl-glm-likelihood-train
    and the three GLM segment datasets, on GPU T4 x2.
    """
    return GatedGLMBlend(build_pool_two_deberta_blend)


def build_pool_glm_balanced_blend():
    """The GLM blend with its test predictions balanced over the 18 models
    (src/models/balance.py): 0.647 on train.csv against 0.626, if the test set,
    like train.csv, has about as many passages of each model. Same inputs as
    pool_glm_blend.
    """
    return Balanced(build_pool_glm_blend)


MODELS = {
    "tfidf_logreg": build_tfidf_logreg,
    "style_logreg": build_style_logreg,
    "tfidf_style_blend": build_tfidf_style_blend,
    "deberta_small": build_deberta_small,
    "deberta_base": build_deberta_base,
    "tfidf_style_deberta_blend": build_tfidf_style_deberta_blend,
    "llm_logreg": build_llm_logreg,
    "full_blend": build_full_blend,
    "pool_tfidf_logreg": build_pool_tfidf_logreg,
    "pool_blend": build_pool_blend,
    "deberta_pool": SavedEncoder,
    "pool_deberta_blend": build_pool_deberta_blend,
    "pool_two_deberta_blend": build_pool_two_deberta_blend,
    "pool_qwen_lora_blend": build_pool_qwen_lora_blend,
    "qwen_lora_pool": load_qwen_lora_pool,
    "pool_qwen_lora_light_blend": build_pool_qwen_lora_light_blend,
    "pool_glm_blend": build_pool_glm_blend,
    "pool_glm_balanced_blend": build_pool_glm_balanced_blend,
}

# Kaggle inputs each model needs besides the competition data (see
# src.predict.find_inputs): the open pool, an open language model, the
# deberta-v3-base weights, or an encoder saved by a train_deberta_pool*
# notebook ("encoder:<name>" for one of several attached).
INPUTS = {
    "deberta_small": ("deberta",),
    "deberta_base": ("deberta",),
    "tfidf_style_deberta_blend": ("deberta",),
    "llm_logreg": ("llm",),
    "full_blend": ("deberta", "llm"),
    "pool_tfidf_logreg": ("pool",),
    "pool_blend": ("pool", "deberta", "llm"),
    "deberta_pool": ("encoder",),
    "pool_deberta_blend": ("pool", "llm", "encoder"),
    "pool_two_deberta_blend": ("pool", "llm", "encoder:deberta_pool_long", "encoder:deberta_pool_big"),
    "pool_qwen_lora_blend": (
        "pool", "llm", "encoder:deberta_pool_long", "encoder:deberta_pool_big", "encoder:qwen_lora_pool",
    ),
    "qwen_lora_pool": ("llm", "encoder:qwen_lora_pool"),
    "pool_qwen_lora_light_blend": (
        "pool", "llm", "encoder:deberta_pool_long", "encoder:deberta_pool_big", "encoder:qwen_lora_pool",
    ),
    "pool_glm_blend": (
        "pool", "llm", "encoder:deberta_pool_long", "encoder:deberta_pool_big", "segrun",
        "glm_segments:glm53_flash", "glm_segments:glm53", "glm_train:glm53_flash", "glm_train:glm53",
    ),
}
INPUTS["pool_glm_balanced_blend"] = INPUTS["pool_glm_blend"]

# Models too heavy for a quick test: they fine-tune or run a pretrained
# network, or train on the open pool. Run them on Kaggle.
NEURAL = {
    "deberta_small",
    "deberta_base",
    "tfidf_style_deberta_blend",
    "llm_logreg",
    "full_blend",
    "pool_tfidf_logreg",
    "pool_blend",
    "deberta_pool",
    "pool_deberta_blend",
    "pool_two_deberta_blend",
    "pool_qwen_lora_blend",
    "qwen_lora_pool",
    "pool_qwen_lora_light_blend",
    "pool_glm_blend",
    "pool_glm_balanced_blend",
}


In [ ]:
%%writefile src/prompts.py
"""Using the structure of the data: every model answers each prompt at most once.

In train.csv no prompt has two passages from the same model. If the test set
is built the same way, the predictions for one prompt should all be different
models. The test set has no prompt_id, so we first group passages by topic
(passages answering one prompt share vocabulary), then pick the best labelling
of each group in which no model repeats.

Whether the hidden test set follows this structure is an assumption; the
leaderboard is the only way to check it.
"""

import numpy as np
from scipy.optimize import linear_sum_assignment
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.text import TfidfVectorizer

EPS = 1e-12


def cluster_prompts(texts, threshold=0.90):
    """Group passages that seem to answer the same prompt; returns a group id per passage.

    Average-linkage clustering on cosine distance between TF-IDF vectors with
    stop words removed, so only topic words count. Merging stops at `threshold`,
    so the number of prompts doesn't have to be known. 0.90 was chosen on
    train.csv, where the true prompts are known.
    """
    texts = list(texts)
    if len(texts) < 2:
        return np.zeros(len(texts), dtype=int)
    vectorizer = TfidfVectorizer(sublinear_tf=True, stop_words="english", min_df=1)
    vectors = vectorizer.fit_transform(texts).toarray()
    # A passage with no topic words has a zero vector, which has no cosine distance.
    empty = ~vectors.any(axis=1)
    vectors[empty, 0] = 1.0
    clustering = AgglomerativeClustering(
        n_clusters=None, distance_threshold=threshold, metric="cosine", linkage="average"
    )
    return clustering.fit_predict(vectors)


def assign_within_groups(probs, groups):
    """Label each group so that no class repeats, maximising total log-probability.

    This is the assignment problem, solved exactly by the Hungarian algorithm.
    A group larger than the number of classes can't be a single prompt (the
    clustering merged several), so there each class may be used once per
    merged prompt: ceil(group size / number of classes) times.
    """
    probs = np.asarray(probs)
    groups = np.asarray(groups)
    n_classes = probs.shape[1]
    log_probs = np.log(probs + EPS)
    predicted = np.zeros(len(probs), dtype=int)
    for group in np.unique(groups):
        rows = np.where(groups == group)[0]
        copies = int(np.ceil(len(rows) / n_classes))
        row_idx, col_idx = linear_sum_assignment(-np.tile(log_probs[rows], (1, copies)))
        predicted[rows[row_idx]] = col_idx % n_classes
    return predicted


In [ ]:
%%writefile src/predict.py
"""Fit a model on all of train.csv and write submission.parquet.

    python -m src.predict --model tfidf_style_blend --data-dir data

This is what the Kaggle submission notebooks run.
"""

import argparse

import pandas as pd

from src.data import LABELS, find_data_dir, load_test, load_train
from src.models import INPUTS, MODELS
from src.prompts import assign_within_groups, cluster_prompts


def check_submission(submission, test):
    """Fail here rather than have Kaggle reject the file."""
    if list(submission.columns) != ["id", "model"]:
        raise ValueError(f"columns must be id, model; got {list(submission.columns)}")
    if submission.isna().any().any():
        raise ValueError("submission has null values")
    if submission["id"].duplicated().any() or set(submission["id"]) != set(test["id"]):
        raise ValueError("every test id must appear exactly once")
    unknown = set(submission["model"]) - set(LABELS)
    if unknown:
        raise ValueError(f"labels outside the 18 valid ones: {sorted(unknown)}")


def find_inputs(model):
    """Locate every Kaggle input the model needs, before any slow training.

    A missing input then fails in seconds with a clear message, instead of
    an hour into the run (or invisibly, in the rerun on the hidden test).
    """
    from src.features.glm import find_segments, find_segrun
    from src.features.llm import find_llm
    from src.models.glm_model import find_glm_train
    from src.models.transformer import find_pretrained, find_saved_encoder
    from src.pool import find_pool

    finders = {
        "pool": find_pool, "llm": find_llm, "deberta": find_pretrained, "encoder": find_saved_encoder,
        "segrun": find_segrun,
        "glm_segments": lambda name: f"{len(find_segments(name))} segment files",
        "glm_train": lambda name: find_glm_train(name),
    }  # fmt: skip
    for need in INPUTS.get(model, ()):
        kind, _, name = need.partition(":")  # "encoder:deberta_pool_big" names one encoder
        path = finders[kind](name=name) if name else finders[kind]()
        print(f"input {need}: {path}", flush=True)


def make_submission(model="tfidf_logreg", assign=False, data_dir=None, out="submission.parquet"):
    """With assign=True, group the test passages by prompt and let no model
    repeat inside a group (see src/prompts.py)."""
    data_dir = find_data_dir() if data_dir is None else data_dir
    train, test = load_train(data_dir), load_test(data_dir)
    print(f"train.csv: {len(train)} passages; test.csv: {len(test)} passages", flush=True)
    find_inputs(model)

    fitted = MODELS[model]().fit(train["text"], train["label"])
    # Columns of predict_proba follow fitted.classes_, which are label ids.
    probs = fitted.predict_proba(test["text"])
    if assign:
        groups = cluster_prompts(test["text"])
        columns = assign_within_groups(probs, groups)
        print(f"grouped {len(test)} passages into {groups.max() + 1} prompts")
    else:
        columns = probs.argmax(axis=1)
    predicted = fitted.classes_[columns]

    submission = pd.DataFrame({"id": test["id"], "model": [LABELS[i] for i in predicted]})
    check_submission(submission, test)
    submission.to_parquet(out, index=False)
    print(f"trained on {len(train)} passages, wrote {len(submission)} predictions to {out}")
    return submission


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])
    parser.add_argument("--model", default="tfidf_logreg", choices=sorted(MODELS))
    parser.add_argument("--assign", action="store_true")
    parser.add_argument("--data-dir", default=None)
    parser.add_argument("--out", default="submission.parquet")
    args = parser.parse_args()
    make_submission(args.model, args.assign, args.data_dir, args.out)


In [ ]:
%%writefile src/llm_experiment.py
"""Experiment: how well do frozen-LLM features name the model?

Run on Kaggle with a GPU and an open language model attached:

    from src.llm_experiment import main; main()

It extracts the features of every training passage once, then cross-validates
a logistic regression on each block of features with the project's folds.
Everything lands in the output folder: the features (llm_features.npz), the
out-of-fold probabilities of each variant (oof_llm_<block>.npy) and a table
(llm_experiment.csv). The oof files go into oof/ locally for blending with
src.blend_oof.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.data import LABELS, find_data_dir, load_train, make_folds
from src.evaluate import fold_scores
from src.features.llm import extract, find_llm

C_VALUES = (0.01, 0.1, 1.0)


def cross_validate_block(X, y, folds, C):
    oof = np.zeros((len(y), len(LABELS)))
    for fold in np.unique(folds):
        train, valid = folds != fold, folds == fold
        model = make_pipeline(
            StandardScaler(), LogisticRegression(C=C, max_iter=2000, class_weight="balanced")
        )
        model.fit(X[train], y[train])
        oof[np.ix_(valid, model.classes_)] = model.predict_proba(X[valid])
    return oof


def main(out_dir="."):
    out_dir = Path(out_dir)
    df = load_train(find_data_dir())
    folds = make_folds(df)
    y = df["label"].to_numpy()
    source = find_llm()
    print(f"language model: {source}")

    features = extract(df["text"], source)
    np.savez_compressed(out_dir / "llm_features.npz", **features)

    layer_names = [name for name in features if name != "stats"]
    blocks = dict(features)
    blocks["all_layers"] = np.hstack([features[name] for name in layer_names])
    blocks["all_layers+stats"] = np.hstack([blocks["all_layers"], features["stats"]])

    rows = []
    for name, X in blocks.items():
        best = None
        for C in C_VALUES:
            oof = cross_validate_block(X, y, folds, C)
            scores = fold_scores(y, oof.argmax(axis=1), folds)
            rows.append({"features": name, "n_features": X.shape[1], "C": C,
                         "cv_mean": scores.mean(), "cv_std": scores.std()})  # fmt: skip
            print(f"{name:18s} C={C:<5} {scores.mean():.4f} +- {scores.std():.4f}", flush=True)
            if best is None or scores.mean() > best[0]:
                best = (scores.mean(), oof)
        np.save(out_dir / f"oof_llm_{name.replace('+', '_')}.npy", best[1])

    table = pd.DataFrame(rows)
    table.to_csv(out_dir / "llm_experiment.csv", index=False)
    return table


In [ ]:
%%writefile src/pool_experiment.py
"""Experiment: how much is the public open pool worth to our cheap models?

Run on Kaggle (CPU is enough) with the open pool dataset attached:

    from src.pool_experiment import main; main()

Each variant trains the TF-IDF model and the style model on a sample of the
pool, with no passage from train.csv, and is scored on all of train.csv. The
pool has no answers to train.csv's prompts, so this is a held-out test and
needs no folds. The last variants add the real training passages, so they
are cross-validated with the project's folds.

Outputs: pool_experiment.csv (the table) and probs_<variant>_<model>.npy
(class probabilities for every train.csv passage, for blending locally).
"""

import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

from src.data import LABELS, find_data_dir, load_train, make_folds
from src.models.linear import build_style_logreg
from src.models.pool_model import build_big_tfidf_logreg
from src.pool import load_pool, sample_pool

EPS = 1e-12
# name -> (passages per label, typography-matched sampling)
VARIANTS = {
    "pool_2500": (2500, False),
    "pool_2500_matched": (2500, True),
    "pool_8000": (8000, False),
}
CV_PER_LABEL = 2500  # pool passages per label added to each training fold


def fit_predict(train_texts, train_labels, test_texts):
    """Fit both models; return {model name: probabilities with 18 columns}."""
    out = {}
    for name, build in (("tfidf", build_big_tfidf_logreg), ("style", build_style_logreg)):
        model = build().fit(train_texts, train_labels)
        probs = np.zeros((len(test_texts), len(LABELS)))
        probs[:, model.classes_] = model.predict_proba(test_texts)
        out[name] = probs
    out["blend"] = np.exp(np.log(out["tfidf"] + EPS) + 0.5 * np.log(out["style"] + EPS))
    return out


def main(out_dir="."):
    out_dir = Path(out_dir)
    df = load_train(find_data_dir())
    y = df["label"].to_numpy()
    pool = load_pool()
    print(f"pool: {len(pool)} passages; train.csv: {len(df)}", flush=True)
    rows = []

    def record(variant, n_train, probs_by_model, started):
        for model, probs in probs_by_model.items():
            score = balanced_accuracy_score(y, probs.argmax(axis=1))
            rows.append({"variant": variant, "model": model, "n_train": n_train, "score": score})
            np.save(out_dir / f"probs_{variant}_{model}.npy", probs / probs.sum(axis=1, keepdims=True))
            print(f"{variant:22s} {model:6s} {score:.4f}", flush=True)
        print(f"  ({variant} took {time.time() - started:.0f}s)", flush=True)

    # 1. Pool only, scored on all of train.csv.
    for variant, (per_label, matched) in VARIANTS.items():
        started = time.time()
        sample = sample_pool(pool, per_label, matched)
        record(variant, len(sample), fit_predict(sample["text"], sample["label"], df["text"]), started)

    # 2. Train.csv alone, and train.csv plus pool, cross-validated on train.csv.
    folds = make_folds(df)
    extra = sample_pool(pool, CV_PER_LABEL)
    for variant, add_pool in (("train_only_cv", False), ("train_plus_pool_cv", True)):
        started = time.time()
        oof = {name: np.zeros((len(df), len(LABELS))) for name in ("tfidf", "style", "blend")}
        for fold in np.unique(folds):
            fit_rows = df[folds != fold]
            texts, labels = fit_rows["text"], fit_rows["label"]
            if add_pool:
                texts = pd.concat([texts, extra["text"]])
                labels = pd.concat([labels, extra["label"]])
            for name, probs in fit_predict(texts, labels, df["text"][folds == fold]).items():
                oof[name][folds == fold] = probs
        n_train = int((folds != 0).sum()) + (len(extra) if add_pool else 0)
        record(variant, n_train, oof, started)

    table = pd.DataFrame(rows)
    table.to_csv(out_dir / "pool_experiment.csv", index=False)
    return table


In [ ]:
%%writefile src/pool_encoder.py
"""Fine-tune DeBERTa-v3-base on the public open pool and save it.

Run on Kaggle with a GPU, the deberta-v3-base weights and the open pool
attached:

    from src.pool_encoder import main; main()

The model learns from pool passages only, never from train.csv, so scoring it
on all of train.csv is a held-out test (the pool has no answers to
train.csv's prompts). Outputs:

- <name>/: the trained model. Attach this notebook's output to a
  submission notebook, where SavedEncoder loads it instead of training.
- probs_<name>.npy: class probabilities for every train.csv passage, for
  choosing blend weights locally.
- <name>.csv: the score.

`name` is deberta_pool by default. build=build_pool_qwen_lora trains a LoRA
on Qwen 3 4B instead (attach Qwen 3 instead of deberta-v3-base).
"""

import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

from src.data import LABELS, find_data_dir, load_train
from src.models.transformer import FineTunedEncoder
from src.pool import load_pool, sample_pool

PER_LABEL = 4000  # about 70,000 passages; the smallest class has about 3,500


def build_pool_encoder(**kwargs):
    settings = dict(
        epochs=2,
        lr=3e-5,
        batch_size=16,
        max_length=384,  # the longest train.csv passage is 351 tokens
        fast=True,
        verbose=True,
        progress_every=250,
        time_limit_hours=9,  # Kaggle stops a run at 12 hours
    )
    settings.update(kwargs)
    return FineTunedEncoder("microsoft/deberta-v3-base", **settings)


def build_pool_qwen_lora(**kwargs):
    """Qwen 3 4B, first 18 of its 36 layers, with a rank-16 LoRA on every linear layer.

    One epoch: each step costs far more than with DeBERTa. The loss and the
    time left are printed every 100 steps; if the run would pass
    time_limit_hours it stops there and still saves.
    """
    from src.features.llm import find_llm

    settings = dict(
        epochs=1,
        lr=2e-4,
        head_lr_factor=5,
        batch_size=16,
        max_length=384,
        fast=True,
        verbose=True,
        progress_every=100,
        time_limit_hours=8,
        balanced=True,
        lora_rank=16,
        lora_alpha=32,
        keep_layers=18,
        gradient_checkpointing=True,
    )
    settings.update(kwargs)
    pretrained = settings.pop("pretrained", None) or find_llm()
    return FineTunedEncoder(pretrained, **settings)


def main(out_dir=".", per_label=PER_LABEL, matched=True, name="deberta_pool", build=None, **encoder_settings):
    out_dir = Path(out_dir)
    df = load_train(find_data_dir())
    sample = sample_pool(load_pool(), per_label, matched)
    print(f"training on {len(sample)} pool passages; testing on {len(df)} train.csv passages", flush=True)

    started = time.time()
    model = (build or build_pool_encoder)(**encoder_settings).fit(sample["text"], sample["label"])
    minutes = (time.time() - started) / 60
    model.save(out_dir / name)  # before scoring, so the weights survive a later error
    print(f"trained in {minutes:.0f} min; saved to {out_dir / name}", flush=True)

    probs = np.zeros((len(df), len(LABELS)))
    probs[:, model.classes_] = model.predict_proba(df["text"])
    np.save(out_dir / f"probs_{name}.npy", probs)
    score = balanced_accuracy_score(df["label"], probs.argmax(axis=1))
    print(f"balanced accuracy on train.csv: {score:.4f}", flush=True)

    table = pd.DataFrame(
        [{"model": name, "n_train": len(sample), "minutes": round(minutes), "score": score}]
    )
    table.to_csv(out_dir / f"{name}.csv", index=False)
    return table


In [ ]:
%%writefile src/models/family_model.py
"""Family specialists: one model per maker, trained only on that maker's models.

Most of the blend's remaining errors are within a family: gemini-3.7-flash is
called gemini-3.8-flash or gemini-3.6-flash, the three GPT-5.6 variants are
called each other. On train.csv, resolving the GPT family perfectly would lift
the GLM blend from 0.626 to 0.717 and the Gemini family to 0.699, so how the
blend splits a family's probability among its members matters more than which
family it picks, which it mostly gets right.

A specialist learns only that split, from open-pool passages of the family's
models. `refine` keeps the blend's total probability for the family and
re-splits it among the members with the specialist's opinion added, so a
passage the blend gives to another maker is left alone.
"""

import numpy as np

from src.data import LABELS

EPS = 1e-12

# family -> label ids, in increasing order (the column order of a specialist's probabilities)
FAMILIES = {
    name: sorted(LABELS.index(label) for label in labels)
    for name, labels in {
        "gpt": ["gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra", "gpt-6-astra"],
        "gemini": ["gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash"],
        "claude": ["claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5", "claude-sonnet-5"],
    }.items()
}


def refine(probs, family, specialist, weight):
    """Re-split each passage's probability for `family` with the specialist's added at `weight`.

    probs: (n, 18) probabilities of the blend; specialist: (n, k) probabilities
    over the family's labels in FAMILIES order. The family's total stays the
    same, so only the choice among its members can change.
    """
    ids = FAMILIES[family]
    out = probs.copy()
    total = probs[:, ids].sum(axis=1, keepdims=True)
    logits = np.log(probs[:, ids] + EPS) + weight * np.log(specialist + EPS)
    logits -= logits.max(axis=1, keepdims=True)
    split = np.exp(logits)
    out[:, ids] = total * split / split.sum(axis=1, keepdims=True)
    return out


In [ ]:
%%writefile src/family_experiment.py
"""Experiment: family specialists trained on the open pool (src/models/family_model.py).

Run on Kaggle with a GPU, deberta-v3-base and the open pool attached:

    from src.family_experiment import main; main()

For each family (GPT, Gemini, Claude) it trains, on open-pool passages of
that family's models only:

- DeBERTa-v3-base, 2 epochs, loss weighted per model, saved as family_<name>/
  so a submission can load it;
- TF-IDF and style logistic regressions, which are cheap enough to refit in a
  submission.

Each is scored on all of train.csv: the pool has no answers to train.csv's
prompts, so this is a held-out test. The printed score is balanced accuracy
among the family's own train.csv passages, choosing only among its members.
probs_family_<name>_<model>.npy holds each specialist's probabilities for
every train.csv passage (columns in FAMILIES order), for src.blend_oof-style
tuning locally against the blend's out-of-fold predictions.
"""

import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

from src.data import find_data_dir, load_train
from src.models.family_model import FAMILIES
from src.models.linear import build_style_logreg
from src.models.pool_model import build_big_tfidf_logreg
from src.models.transformer import free_gpu
from src.pool import load_pool, sample_pool
from src.pool_encoder import build_pool_encoder

DEBERTA_PER_LABEL = 8000
LINEAR_PER_LABEL = 15000


def family_score(y, probs, ids):
    """Balanced accuracy on the family's passages, choosing among its members only."""
    rows = np.isin(y, ids)
    predicted = np.asarray(ids)[probs[rows].argmax(axis=1)]
    return balanced_accuracy_score(y[rows], predicted)


def main(out_dir=".", families=tuple(FAMILIES), deberta=True, linear=True):
    out_dir = Path(out_dir)
    df = load_train(find_data_dir())
    y = df["label"].to_numpy()
    pool = load_pool()
    rows = []

    def record(family, model, probs, n_train, started):
        np.save(out_dir / f"probs_family_{family}_{model}.npy", probs)
        score = family_score(y, probs, FAMILIES[family])
        minutes = (time.time() - started) / 60
        rows.append({"family": family, "model": model, "n_train": n_train, "minutes": round(minutes),
                     "family_score": score})  # fmt: skip
        print(f"{family:7s} {model:8s} {score:.4f} ({n_train} passages, {minutes:.0f} min)", flush=True)

    for family in families:
        ids = FAMILIES[family]
        members = pool[pool["label"].isin(ids)]
        counts = members["model"].value_counts().to_dict()
        print(f"{family}: pool passages per model {counts}", flush=True)

        if deberta:
            started = time.time()
            sample = sample_pool(members, DEBERTA_PER_LABEL, matched=True)
            model = build_pool_encoder(epochs=2, balanced=True, time_limit_hours=2.5)
            model.fit(sample["text"], sample["label"])
            assert list(model.classes_) == ids
            model.save(out_dir / f"family_{family}")
            record(family, "deberta", model.predict_proba(df["text"]), len(sample), started)
            del model
            free_gpu()

        if linear:
            sample = sample_pool(members, LINEAR_PER_LABEL)
            for name, build in (("tfidf", build_big_tfidf_logreg), ("style", build_style_logreg)):
                started = time.time()
                model = build().fit(sample["text"], sample["label"])
                assert list(model.classes_) == ids
                record(family, name, model.predict_proba(df["text"]), len(sample), started)

    table = pd.DataFrame(rows)
    table.to_csv(out_dir / "family_experiment.csv", index=False)
    return table


In [ ]:
%%writefile src/segrun/segio.py
# Copied unchanged from ntl-segrun by José Padilla (estopadilla), published in the public
# competition notebook https://www.kaggle.com/code/estopadilla/ntl-glm-likelihood-train
# (shared code is open-source licensed under section 6b of the competition rules).
"""Read and write the binary files of ntl-segrun (see the header of tools/ntl-segrun/ntl-segrun.cpp)."""
import struct

import numpy as np


def write_texts(path, texts):
    with open(path, 'wb') as f:
        f.write(b'NTLT')
        f.write(struct.pack('<I', len(texts)))
        for t in texts:
            b = t.encode('utf-8')
            f.write(struct.pack('<I', len(b)))
            f.write(b)


def write_tokens(path, seqs):
    """seqs: list of (ids, n_prefix)."""
    with open(path, 'wb') as f:
        f.write(b'NTLK')
        f.write(struct.pack('<I', len(seqs)))
        for ids, n_prefix in seqs:
            ids = np.asarray(ids, dtype='<i4')
            f.write(struct.pack('<II', len(ids), n_prefix))
            f.write(ids.tobytes())


def read_tokens(path):
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLK'
        (n,) = struct.unpack('<I', f.read(4))
        for _ in range(n):
            nt, npre = struct.unpack('<II', f.read(8))
            ids = np.frombuffer(f.read(4 * nt), dtype='<i4')
            out.append((ids, npre))
    return out


def read_scores(path):
    """List of float32 arrays [n_scored, 4]: logprob, entropy, E[logp^2], rank."""
    out = []
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLP'
        n, nf = struct.unpack('<II', f.read(8))
        for _ in range(n):
            (ns,) = struct.unpack('<I', f.read(4))
            out.append(np.frombuffer(f.read(4 * ns * nf), dtype='<f4').reshape(ns, nf))
    return out


def read_state(path):
    with open(path, 'rb') as f:
        assert f.read(4) == b'NTLS'
        dt, w, nt, nr = struct.unpack('<IIIQ', f.read(20))
        dtype = {0: '<f4', 1: '<f2'}.get(dt)
        if dtype is None:  # bf16
            raw = np.frombuffer(f.read(2 * w * nr), dtype='<u2').astype('<u4') << 16
            return raw.view('<f4').reshape(nr, w)
        return np.frombuffer(f.read(np.dtype(dtype).itemsize * w * nr), dtype=dtype).reshape(nr, w).astype('f4')


In [ ]:
%%writefile src/segrun/features.py
# Copied unchanged from ntl-segrun by José Padilla (estopadilla), published in the public
# competition notebook https://www.kaggle.com/code/estopadilla/ntl-glm-likelihood-train
# (shared code is open-source licensed under section 6b of the competition rules).
"""Per-text likelihood features from the per-token scores of ntl-segrun.

Per scored token: lp = log-prob of the true token, H = entropy of the next-token distribution,
m2 = E_p[logp^2] (so Var_p[logp] = m2 - H^2), rank = how many tokens beat the true one (0 = top-1).
"""
import numpy as np
import pandas as pd


def one(s, n_chars):
    lp, h, m2, rank = s[:, 0].astype(np.float64), s[:, 1].astype(np.float64), s[:, 2].astype(np.float64), s[:, 3]
    n = len(lp)
    var = np.maximum(m2 - h * h, 1e-9)
    half = max(n // 2, 1)
    q = np.quantile(lp, [0.05, 0.10, 0.25, 0.50, 0.75, 0.90]) if n else np.zeros(6)
    return {
        'n_tok': n,
        'lp_mean': lp.mean(),
        'lp_std': lp.std(),
        'lp_q05': q[0], 'lp_q10': q[1], 'lp_q25': q[2], 'lp_q50': q[3], 'lp_q75': q[4], 'lp_q90': q[5],
        'lp_min': lp.min(),
        'ent_mean': h.mean(),
        'ent_std': h.std(),
        # Fast-DetectGPT analytic curvature: how far the text's log-likelihood sits above the model's own expectation
        'fdg': (lp + h).sum() / np.sqrt(var.sum()),
        'surprise_mean': (-lp - h).mean(),          # observed surprisal minus expected surprisal
        'z_mean': ((lp + h) / np.sqrt(var)).mean(),
        'logrank_mean': np.log1p(rank).mean(),
        'top1': (rank == 0).mean(),
        'top5': (rank < 5).mean(),
        'top20': (rank < 20).mean(),
        'rank_ge100': (rank >= 100).mean(),
        'lp_mean_first': lp[:half].mean(),
        'lp_mean_second': lp[half:].mean() if n > half else lp.mean(),
        'bpc': -lp.sum() / max(n_chars, 1) / np.log(2),
        'tok_per_char': n / max(n_chars, 1),
    }


def text_features(scores, texts):
    return pd.DataFrame([one(s, len(t)) for s, t in zip(scores, texts)])


In [ ]:
%%writefile src/segrun/run_pipeline.py
# Copied unchanged from ntl-segrun by José Padilla (estopadilla), published in the public
# competition notebook https://www.kaggle.com/code/estopadilla/ntl-glm-likelihood-train
# (shared code is open-source licensed under section 6b of the competition rules).
"""Score texts under a big GLM model that ships as layer-segment GGUFs, with ntl-segrun.

Every text goes through segment k before segment k+1 is loaded, so each weight is read once per chunk.
The residual state between segments lives in one file on local disk, rewritten in place.
If the disk cannot hold the state of all texts, the texts are split into chunks (each chunk reads the
model again).

    python run_pipeline.py --bin BIN_DIR --segs 'GLOB' --csv train.csv --out OUT_DIR --tag flash
        [--work /tmp/ntl] [--prefix STR] [--max-text-tokens 320] [--seq-batch 8] [--state-dtype auto]
        [--limit N] [--deadline-s S] [--pack] [--ubatch-div P] [--budget-s S]

--pack sorts the texts by token count (longest first) and packs them into padded decodes (see ntl-segrun.cpp);
the scores are put back in the input order. --budget-s is a hard limit for the whole call: every ntl-segrun call
gets the time left as its timeout, and after each segment the time of the rest is projected from the segments
done so far; if it does not fit, the call stops with exit code 3 (BudgetExceeded) and writes nothing.

Writes OUT_DIR/{tag}_scores.npz (per-token logprob/entropy/m2/rank, offsets, ids), {tag}_features.csv
and {tag}_timing.json.
"""
import argparse
import glob
import json
import os
import re
import shutil
import struct
import subprocess
import sys
import time

import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import features  # noqa: E402
import segio  # noqa: E402

CHAT_PREFIX = '[gMASK]<sop><|user|><|assistant|><think></think>'


def seg_index(path):
    m = re.search(r'seg(\d+)-of-(\d+)', os.path.basename(path))
    assert m, path
    return int(m.group(1)), int(m.group(2))


def gguf_kv_u32(path, key):
    """Read one uint32 metadata value from a GGUF header without the gguf package."""
    with open(path, 'rb') as f:
        assert f.read(4) == b'GGUF'
        _ver, _nt, nkv = struct.unpack('<IQQ', f.read(20))

        def rstr():
            (n,) = struct.unpack('<Q', f.read(8))
            return f.read(n).decode('utf-8', 'replace')

        sizes = {0: 1, 1: 1, 2: 2, 3: 2, 4: 4, 5: 4, 6: 4, 7: 1, 10: 8, 11: 8, 12: 8}

        def skip(t):
            if t == 8:
                rstr()
            elif t == 9:
                it, n = struct.unpack('<IQ', f.read(12))
                if it in sizes:
                    f.seek(sizes[it] * n, 1)
                else:
                    for _ in range(n):
                        skip(it)
            else:
                f.seek(sizes[t], 1)

        for _ in range(nkv):
            k = rstr()
            (t,) = struct.unpack('<I', f.read(4))
            if k == key:
                assert t == 4, (key, t)
                return struct.unpack('<I', f.read(4))[0]
            skip(t)
    return None


class BudgetExceeded(RuntimeError):
    """The time budget (--budget-s) is not enough: stop and let the caller fall back."""


class EarlyFailure(RuntimeError):
    """ntl-segrun failed before writing any output (model load or context reservation, e.g. out of GPU memory)."""


def run(cmd, log, timeout=None):
    t0 = time.time()
    try:
        p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, errors='replace',
                           timeout=timeout)
    except subprocess.TimeoutExpired as e:
        out = e.stdout.decode('utf-8', 'replace') if isinstance(e.stdout, bytes) else (e.stdout or '')
        with open(log, 'a') as lf:
            lf.write('$ ' + ' '.join(cmd) + '\n' + out + '\n[killed: timeout]\n')
        raise BudgetExceeded(f'timeout after {timeout:.0f} s: {" ".join(cmd)}')
    with open(log, 'a') as lf:
        lf.write('$ ' + ' '.join(cmd) + '\n' + p.stdout)
    dt = time.time() - t0
    if p.returncode != 0:
        print('\n'.join(p.stdout.splitlines()[-40:]), flush=True)
        err = EarlyFailure if 'context ready' not in p.stdout else RuntimeError
        raise err(f'failed ({p.returncode}): {" ".join(cmd)}')
    return dt


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--bin', required=True)
    ap.add_argument('--segs', required=True, help='glob of the segment files')
    ap.add_argument('--csv', required=True)
    ap.add_argument('--text-col', default='text')
    ap.add_argument('--id-col', default='id')
    ap.add_argument('--out', required=True)
    ap.add_argument('--tag', required=True)
    ap.add_argument('--work', default='/tmp/ntl')
    ap.add_argument('--prefix', default=CHAT_PREFIX)
    ap.add_argument('--max-text-tokens', type=int, default=320)
    ap.add_argument('--seq-batch', type=int, default=8)
    ap.add_argument('--state-dtype', default='auto', help='auto | f32 | f16 | bf16')
    ap.add_argument('--threads', type=int, default=min(4, os.cpu_count() or 4))
    ap.add_argument('--limit', type=int, default=-1)
    ap.add_argument('--disk-frac', type=float, default=0.8, help='share of the free work disk the state may use')
    ap.add_argument('--deadline-s', type=float, default=0, help='stop before starting a chunk after this many seconds')
    ap.add_argument('--pack', action='store_true', help='sort by length and pack padded decodes (ntl-segrun --pack)')
    ap.add_argument('--ubatch-div', type=int, default=1, help='n_ubatch = n_batch / P (pipeline parallel on 2 GPUs)')
    ap.add_argument('--budget-s', type=float, default=0, help='hard time limit of the whole call (0 = none)')
    a = ap.parse_args()
    try:
        main_run(a)
    except BudgetExceeded as e:
        print(f'[{a.tag}] BUDGET EXCEEDED: {e}', flush=True)
        sys.exit(3)


def main_run(a):

    t_start = time.time()

    def left():
        return None if not a.budget_s else a.budget_s - (time.time() - t_start)

    os.makedirs(a.out, exist_ok=True)
    os.makedirs(a.work, exist_ok=True)
    log = os.path.join(a.out, f'{a.tag}_run.log')
    segrun = os.path.join(a.bin, 'ntl-segrun')

    segs = sorted(glob.glob(a.segs), key=lambda p: seg_index(p)[0])
    assert segs, a.segs
    n_seg = seg_index(segs[0])[1]
    assert [seg_index(p)[0] for p in segs] == list(range(n_seg)), segs
    ranges = [(gguf_kv_u32(p, 'ntl.layer_start'), gguf_kv_u32(p, 'ntl.layer_end')) for p in segs]
    print(f'[{a.tag}] {n_seg} segments, layers {ranges}', flush=True)
    for (s0, e0), (s1, _e1) in zip(ranges, ranges[1:]):
        assert e0 == s1, ranges

    df = pd.read_csv(a.csv)
    if a.limit > 0:
        df = df.iloc[:a.limit]
    texts = df[a.text_col].astype(str).tolist()
    tx = os.path.join(a.work, f'{a.tag}_texts.bin')
    tk = os.path.join(a.work, f'{a.tag}_tok.bin')
    segio.write_texts(tx, texts)
    run([segrun, 'tokenize', '--model', segs[0], '--texts', tx, '--out', tk, '--prefix', a.prefix,
         '--max-text-tokens', str(a.max_text_tokens)], log)
    seqs = segio.read_tokens(tk)
    n_rows = np.array([len(ids) for ids, _ in seqs])
    # processing order: longest first when packing (similar lengths share a decode, little padding)
    order = np.argsort(-n_rows, kind='stable') if a.pack else np.arange(len(seqs))
    print(f'[{a.tag}] {len(seqs)} texts, {n_rows.sum()} rows, max {n_rows.max()}, prefix {seqs[0][1]}', flush=True)

    # state width: glm5-next hc*n_embd, glm-dsa n_embd (read from the first segment's metadata)
    n_embd = gguf_kv_u32(segs[0], 'glm5-next.embedding_length')
    if n_embd is not None:
        width = gguf_kv_u32(segs[0], 'glm5-next.hyper_connection.count') * n_embd
    else:
        width = gguf_kv_u32(segs[0], 'glm-dsa.embedding_length')
    free = shutil.disk_usage(a.work).free
    if a.state_dtype == 'auto':
        # exact f32 if the whole state fits in one chunk, else bf16 (half the disk, no overflow risk)
        a.state_dtype = 'f32' if n_rows.sum() * width * 4 <= free * a.disk_frac else 'bf16'
        print(f'[{a.tag}] state dtype auto -> {a.state_dtype}', flush=True)
    el = 2 if a.state_dtype in ('f16', 'bf16') else 4
    max_rows = int(free * a.disk_frac / (width * el))
    chunks, cur, cur_rows = [], [], 0
    for i in order:
        r = n_rows[i]
        if cur and cur_rows + r > max_rows:
            chunks.append(cur)
            cur, cur_rows = [], 0
        cur.append(i)
        cur_rows += r
    chunks.append(cur)
    print(f'[{a.tag}] state width {width}, {el} B/value, {free/2**30:.1f} GiB free on {a.work} -> '
          f'{len(chunks)} chunk(s)', flush=True)

    multi_gpu = False
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
        multi_gpu = out.count('GPU ') > 1
    except FileNotFoundError:
        pass

    timing = {'tag': a.tag, 'n_texts': len(seqs), 'n_rows': int(n_rows.sum()), 'chunks': [], 'segments': segs,
              'ranges': ranges, 'seq_batch': a.seq_batch, 'state_dtype': a.state_dtype, 'prefix': a.prefix,
              'pack': a.pack, 'ubatch_div': a.ubatch_div, 'budget_s': a.budget_s}
    all_scores = [None] * len(seqs)
    st = os.path.join(a.work, f'{a.tag}_state.bin')
    done = 0
    for ci, idx in enumerate(chunks):
        if a.deadline_s and time.time() - t_start > a.deadline_s:
            print(f'[{a.tag}] deadline reached before chunk {ci}, stopping', flush=True)
            break
        ctk = os.path.join(a.work, f'{a.tag}_tok_c{ci}.bin')
        segio.write_tokens(ctk, [seqs[i] for i in idx])
        sc = os.path.join(a.work, f'{a.tag}_scores_c{ci}.bin')
        seg_t = []
        for k, p in enumerate(segs):
            while True:
                cmd = [segrun, 'run', '--model', p, '--tokens', ctk, '--threads', str(a.threads),
                       '--seq-batch', str(a.seq_batch), '--state-dtype', a.state_dtype]
                if multi_gpu:
                    cmd.append('--split-mid')
                if a.pack:
                    cmd.append('--pack')
                if a.ubatch_div > 1:
                    cmd += ['--ubatch-div', str(a.ubatch_div)]
                if k > 0:
                    cmd += ['--state-in', st]
                cmd += ['--out', sc if k == n_seg - 1 else st]
                if left() is not None and left() <= 0:
                    raise BudgetExceeded('no time left')
                try:
                    dt = run(cmd, log, timeout=left())
                    break
                except EarlyFailure:
                    # nothing written yet (the input state is intact): retry with half the texts per decode
                    if a.seq_batch == 1:
                        raise
                    a.seq_batch //= 2
                    timing['seq_batch_reduced_to'] = a.seq_batch
                    print(f'[{a.tag}] segment {k+1} failed before writing: retrying with seq-batch {a.seq_batch}', flush=True)
            seg_t.append(dt)
            print(f'[{a.tag}] chunk {ci+1}/{len(chunks)} segment {k+1}/{n_seg} {os.path.basename(p)}: {dt:.0f} s '
                  f'(total {time.time()-t_start:.0f} s)', flush=True)
            if a.budget_s:
                # projection: the remaining segments of this chunk at the mean so far (the last one, with the
                # softmax over the vocabulary, counts double), plus the other chunks at this chunk's pace
                rest = n_seg - k - 1
                per = float(np.mean(seg_t))
                proj = per * (rest + (1 if rest > 0 else 0)) + per * n_seg * 1.15 * (len(chunks) - ci - 1)
                if proj > left():
                    raise BudgetExceeded(f'projected {proj:.0f} s for the rest, {left():.0f} s left')
        for i, s in zip(idx, segio.read_scores(sc)):
            all_scores[i] = s
        done += len(idx)
        timing['chunks'].append({'n_texts': len(idx), 'seg_seconds': seg_t})
        for f in (ctk, sc, st):
            if os.path.exists(f):
                os.remove(f)

    ok = [i for i, s in enumerate(all_scores) if s is not None]
    off = np.zeros(len(ok) + 1, dtype=np.int64)
    off[1:] = np.cumsum([len(all_scores[i]) for i in ok])
    np.savez_compressed(os.path.join(a.out, f'{a.tag}_scores.npz'),
                        ids=df[a.id_col].astype(str).values[ok],
                        scores=np.concatenate([all_scores[i] for i in ok]).astype(np.float32),
                        offsets=off,
                        n_prefix=np.array([seqs[i][1] for i in ok]),
                        tokens=np.concatenate([seqs[i][0][seqs[i][1]:] for i in ok]).astype(np.int32))
    feats = features.text_features([all_scores[i] for i in ok], [texts[i] for i in ok])
    feats.insert(0, a.id_col, df[a.id_col].astype(str).values[ok])
    feats.to_csv(os.path.join(a.out, f'{a.tag}_features.csv'), index=False)
    timing['total_s'] = time.time() - t_start
    timing['n_done'] = len(ok)
    with open(os.path.join(a.out, f'{a.tag}_timing.json'), 'w') as f:
        json.dump(timing, f, indent=1)
    print(f'[{a.tag}] done: {len(ok)}/{len(seqs)} texts in {timing["total_s"]:.0f} s', flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile src/features/glm.py
"""Likelihood of each passage under two of the 18 models themselves: GLM-5.3-Flash and GLM-5.3.

A model gives its own text higher probability than other models' text, so
"how likely is this passage under GLM" separates the two GLM labels from the
rest, and from each other, better than style does. In our blend they are the
weakest labels (glm-5.3 recall 0.20, glm-5.3-flash 0.37), mostly mistaken for
claude-fable-5 and for each other.

The open GLM weights are far too big for Kaggle's two 15 GB GPUs (GLM-5.3-Flash
alone is 118 GB compressed), so they come as public datasets of layer-segment
GGUF files, and a patched llama.cpp, ntl-segrun, runs every passage through one
segment at a time, keeping the state between segments on disk. ntl-segrun and
its driver scripts (src/segrun/) are José Padilla's, published in the public
competition notebook https://www.kaggle.com/code/estopadilla/ntl-glm-likelihood-train;
its output holds the compiled program, so nothing is built here. We use it as
a tool: what we add is the features' use (src/glm_experiment.py, the
classifier and the blend).

Needs, attached on Kaggle: GPU T4 x2, the output of that notebook (ntl-bin/),
and the segment datasets jamesliebel/glm53-flash-iq3xxs-segs-mirror (6 files)
and jamesliebel/glm53-iq3xxs-segs-a-mirror + -b-mirror (19 files together).
"""

import os
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

ROOT = Path(__file__).resolve().parent.parent.parent
PIPELINE = ROOT / "src" / "segrun" / "run_pipeline.py"

# model tag -> file pattern of its segments
SEGMENTS = {
    "glm53_flash": "flash_iq3xxs-seg*-of-06-*.gguf",
    "glm53": "glm53_iq3xxs-seg*-of-19-*.gguf",
}
# As in Padilla's runs: the GLM chat template with an empty user turn and
# reasoning off, then the passage as the assistant's answer, cut at 320 tokens.
PREFIX = "[gMASK]<sop><|system|>Reasoning Effort: Max<|user|><|assistant|><think></think>"
MAX_TEXT_TOKENS = 320


def find_segrun(root="/kaggle/input"):
    """The ntl-bin folder (ntl-segrun and its CUDA libraries) from the attached notebook output."""
    override = os.environ.get("NTL_SEGRUN_DIR")
    if override:
        return Path(override)
    hits = sorted(Path(root).rglob("ntl-bin/ntl-segrun"))
    if not hits:
        raise FileNotFoundError(
            f"no ntl-bin/ntl-segrun under {root}: attach the output of the notebook "
            "estopadilla/ntl-glm-likelihood-train"
        )
    return hits[0].parent


def find_segments(model, root="/kaggle/input"):
    """The segment files of one model, one per index (a file may come from two mirrors)."""
    found = {}
    for path in sorted(Path(root).rglob(SEGMENTS[model])):
        found.setdefault(path.name, path)
    if not found:
        raise FileNotFoundError(f"no {SEGMENTS[model]} under {root}: attach the {model} segment datasets")
    return [found[name] for name in sorted(found)]


def _prepare(model, work):
    """Copy ntl-segrun to a writable place and link the segments into one folder."""
    source = find_segrun()
    binary = Path(work) / "ntl-bin"
    if not (binary / "ntl-segrun").exists():
        shutil.copytree(source, binary, dirs_exist_ok=True)
        (binary / "ntl-segrun").chmod(0o755)
    segments = Path(work) / f"segs_{model}"
    segments.mkdir(parents=True, exist_ok=True)
    for path in find_segments(model):
        link = segments / path.name
        if not link.exists():
            link.symlink_to(path)
    return binary, segments


def score(texts, model, out_dir, work="/tmp/ntl", ids=None, deadline_s=0, budget_s=0):
    """Likelihood features of each passage under `model`, as a DataFrame in input order.

    One row per passage: mean, spread and quantiles of the log-probability of
    each token, the model's entropy, the rank of the actual token, and the
    Fast-DetectGPT curvature (src/segrun/features.py). Passages the run did
    not reach (deadline) are missing from the result. With `budget_s`, a run
    projected to take longer stops early with an error and writes nothing.
    """
    texts = list(texts)
    ids = [str(i) for i in (range(len(texts)) if ids is None else ids)]
    work, out_dir = Path(work), Path(out_dir)
    work.mkdir(parents=True, exist_ok=True)
    out_dir.mkdir(parents=True, exist_ok=True)
    binary, segments = _prepare(model, work)
    csv = work / f"{model}_input.csv"
    pd.DataFrame({"id": ids, "text": texts}).to_csv(csv, index=False)

    env = dict(os.environ)
    if (binary / "lib").is_dir():  # the CUDA runtime it was built against
        env["LD_LIBRARY_PATH"] = f"{binary / 'lib'}:{env.get('LD_LIBRARY_PATH', '')}"
    command = [
        sys.executable, str(PIPELINE), "--bin", str(binary), "--segs", str(segments / "*.gguf"),
        "--csv", str(csv), "--out", str(out_dir), "--tag", model, "--work", str(work / "state"),
        "--prefix", PREFIX, "--max-text-tokens", str(MAX_TEXT_TOKENS), "--seq-batch", "8",
        "--deadline-s", str(deadline_s), "--budget-s", str(max(budget_s, 0)),
    ]  # fmt: skip
    print("$", " ".join(command), flush=True)
    subprocess.run(command, env=env, check=True)

    features = pd.read_csv(out_dir / f"{model}_features.csv", dtype={"id": str})
    features = features.set_index("id").reindex(ids)
    return features.dropna(how="all")


In [ ]:
%%writefile src/glm_experiment.py
"""Experiment: how well does the likelihood under the GLM models name the model?

Run on Kaggle (GPU T4 x2; inputs in src/features/glm.py):

    from src.glm_experiment import main; main()

Scores every train.csv passage under GLM-5.3-Flash, then GLM-5.3 (about an
hour together), saves the features (glm_<model>_train.csv: the submission
reuses them instead of scoring train.csv again) and cross-validates a
logistic regression on them with the project's folds. The best variant's
out-of-fold probabilities land in probs_glm.npy, for oof/glm.npy and
src.blend_oof.
"""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.data import LABELS, find_data_dir, load_train, make_folds
from src.evaluate import fold_scores
from src.features.glm import SEGMENTS, score

C_VALUES = (0.01, 0.1, 1.0)


def feature_matrix(tables):
    """Features of each model side by side, plus Flash minus GLM-5.3 for each.

    The difference asks the question directly: which of the two GLMs finds
    this passage more likely.
    """
    parts = [table.add_prefix(f"{model}:") for model, table in tables.items()]
    if len(tables) == 2:
        (_, a), (_, b) = tables.items()
        parts.append((a - b).add_prefix("diff:"))
    return pd.concat(parts, axis=1)


def cross_validate(X, y, folds, C):
    oof = np.zeros((len(y), len(LABELS)))
    for fold in np.unique(folds):
        train, valid = folds != fold, folds == fold
        model = make_pipeline(
            StandardScaler(), LogisticRegression(C=C, max_iter=3000, class_weight="balanced")
        )
        model.fit(X[train], y[train])
        oof[np.ix_(valid, model.classes_)] = model.predict_proba(X[valid])
    return oof


def report_glm_recall(y, predicted):
    for name in ("glm-5.3", "glm-5.3-flash", "claude-fable-5"):
        label = LABELS.index(name)
        print(f"  recall {name}: {(predicted[y == label] == label).mean():.2f}", flush=True)


def main(out_dir="."):
    out_dir = Path(out_dir)
    df = load_train(find_data_dir())
    folds = make_folds(df)
    y = df["label"].to_numpy()
    ids = df["id"].astype(str).tolist()

    tables = {}
    for model in SEGMENTS:
        try:
            table = score(df["text"], model, out_dir / "glm_out", ids=ids)
        except Exception as error:  # e.g. GLM-5.3's datasets not attached: go on with Flash
            print(f"{model} failed: {error!r}", flush=True)
            continue
        table.to_csv(out_dir / f"glm_{model}_train.csv")
        print(f"{model}: {len(table)}/{len(df)} passages scored", flush=True)
        tables[model] = table.reindex(ids)
    if not tables:
        raise RuntimeError("no GLM model could be run")

    X_all = feature_matrix(tables)
    complete = X_all.notna().all(axis=1).to_numpy()
    if not complete.all():
        print(f"only {complete.sum()} passages have every feature; scoring on those", flush=True)
    variants = {model: feature_matrix({model: table}) for model, table in tables.items()}
    if len(tables) == 2:
        variants["both"] = X_all

    rows, best = [], None
    for name, X in variants.items():
        X = X.to_numpy()[complete]
        for C in C_VALUES:
            oof = cross_validate(X, y[complete], folds[complete], C)
            scores = fold_scores(y[complete], oof.argmax(axis=1), folds[complete])
            rows.append({"features": name, "n_features": X.shape[1], "C": C,
                         "cv_mean": scores.mean(), "cv_std": scores.std()})  # fmt: skip
            print(f"{name:12s} C={C:<5} {scores.mean():.4f} +- {scores.std():.4f}", flush=True)
            report_glm_recall(y[complete], oof.argmax(axis=1))
            if best is None or scores.mean() > best[0]:
                best = (scores.mean(), name, C, oof)

    probs = np.full((len(df), len(LABELS)), 1.0 / len(LABELS))  # unscored passages: uniform
    probs[complete] = best[3]
    np.save(out_dir / "probs_glm.npy", probs)
    print(f"best: {best[1]} C={best[2]} {best[0]:.4f}; saved probs_glm.npy", flush=True)
    table = pd.DataFrame(rows)
    table.to_csv(out_dir / "glm_experiment.csv", index=False)
    return table


In [ ]:
from src.predict import make_submission

submission = make_submission(model='full_blend', assign=False)
submission["model"].value_counts()